# Imports

In [ ]:
# ============================================================
# ANOM.0) Imports
# ============================================================

from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

# Project helpers (we already have these modules)
from src.bench.guardrails_artifacts import (
    build_guardrail_fn_registry,
    load_guardrail_spec,
)

# Optional (recommended): our parquet reader helper from the project
# If read_pq exists in our notebook utils, import it instead of redefining.
read_pq = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

print("Imports OK.")

# Paths + load V3 artifacts (spec + thresholds)

In [ ]:
# ============================================================
# ANOM.1) Load V3 artifacts (spec + thresholds)
# ============================================================

GUARDRAIL_SPEC_PATH = Path("artifacts/guardrails/v3/guardrail_v3_spec.json")
THRESHOLDS_PATH     = Path("artifacts/guardrails/v3/thresholds_v2.json")

assert GUARDRAIL_SPEC_PATH.exists(), f"Missing: {GUARDRAIL_SPEC_PATH}"
assert THRESHOLDS_PATH.exists(), f"Missing: {THRESHOLDS_PATH}"

registry = build_guardrail_fn_registry()

loaded = load_guardrail_spec(
    spec_path=GUARDRAIL_SPEC_PATH,
    thresholds_path=THRESHOLDS_PATH,
    fn_registry=registry,
)

guardrail_v3_rehydrated = loaded["guardrail"]
thresholds_v2_loaded = loaded["thresholds"]

EPS = float(thresholds_v2_loaded.get("epsilon", 1e-9))

print("Loaded guardrail:", guardrail_v3_rehydrated.get("name"), "| components:", len(guardrail_v3_rehydrated["components"]))
print("Threshold keys:", len(thresholds_v2_loaded))
print("EPS:", EPS)

# Load eval universe

In [ ]:
# ============================================================
# ANOM.2) Load anomaly universe (D/G + V3 scored frame)
# Preferred: load from a parquet we produced in the guardrails notebook
# ============================================================

# Option 1 (recommended): point to an explicit parquet file we saved.
# Update this path once, then keep the notebook stable.
EVAL_SCORED_DG_V3_PATH = Path("artifacts/eval_universe/eval_scored_DG_V3.parquet")

if EVAL_SCORED_DG_V3_PATH.exists():
    eval_scored_DG_V3 = pd.read_parquet(EVAL_SCORED_DG_V3_PATH)
    print("Loaded:", EVAL_SCORED_DG_V3_PATH)
else:
    # Option 2: load from our failure-analysis parquet bundle if that's our current storage.
    # This assumes PARQUET_DIR points to the bundle we used in EVAL.1.
    # If we already have read_pq in this repo, we can use it; otherwise fallback to pd.read_parquet.
    PARQUET_DIR = Path("artifacts/failure_analysis_v2/parquet_rich_v2_20260309_093101")  # update if needed
    candidate = PARQUET_DIR / "failure_df_full_v2.parquet"
    assert candidate.exists(), f"Could not find eval universe parquet at {candidate} or {EVAL_SCORED_DG_V3_PATH}"

    eval_scored_DG_V3 = pd.read_parquet(candidate)
    print("Loaded:", candidate)
    print("NOTE: This is failure_df_full_v2. Confirm it reflects D/G+V3 expected_cost in this bundle.")

print("Rows:", len(eval_scored_DG_V3))
print("Cols:", eval_scored_DG_V3.shape[1])
display(eval_scored_DG_V3.head(3))

# Minimal schema sanity (so later cells fail fast)

In [ ]:
# ============================================================
# ANOM.3) Minimal schema sanity checks (fail fast)
# ============================================================

REQ = [
    "row_id", "Rndrng_NPI", "HCPCS_Cd", "Year",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
    "services", "benes", "expected_cost_support_tier", "has_lag",
    "rbcs_family_desc", "state",
]

missing = [c for c in REQ if c not in eval_scored_DG_V3.columns]
assert not missing, f"eval_scored_DG_V3 is missing required columns: {missing}"

# Numeric coercions (defensive)
for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio","services","benes"]:
    eval_scored_DG_V3[c] = pd.to_numeric(eval_scored_DG_V3[c], errors="coerce")

eval_scored_DG_V3["has_lag"] = eval_scored_DG_V3["has_lag"].astype(bool)

# Basic finiteness checks
assert eval_scored_DG_V3["observed_cost"].notna().all(), "observed_cost has NaNs"
assert eval_scored_DG_V3["expected_cost"].notna().all(), "expected_cost has NaNs"
assert np.isfinite(eval_scored_DG_V3["oe_ratio"].to_numpy(dtype="float64")).all(), "oe_ratio has non-finite values"

print("Schema sanity OK.")

# ANOM.1 Compute confidence flags (new, auditable) + helper percentiles

In [ ]:
# ============================================================
# ANOM.1) Confidence flags + within-slice percentiles
# ============================================================

import numpy as np
import pandas as pd

df = eval_scored_DG_V3.copy()

# -----------------------------
# 1) Define a new "is_high_conf" flag (auditable, stable)
#    Tune thresholds later after looking at distributions.
# -----------------------------
MIN_SERVICES = 50
MIN_BENES = 20
HIGH_CONF_TIERS = {"high", "medium_high"}

df["is_high_conf"] = (
    df["expected_cost_support_tier"].astype(str).isin(HIGH_CONF_TIERS)
    & (df["services"].fillna(0) >= MIN_SERVICES)
    & (df["benes"].fillna(0) >= MIN_BENES)
)

# Optional: hot-start only (uncomment if we want this stricter definition)
# df["is_high_conf"] = df["is_high_conf"] & df["has_lag"].astype(bool)

# Keep our existing flag too
df["high_confidence_anomaly_candidate"] = df["high_confidence_anomaly_candidate"].astype(bool)

print("High-conf counts:")
display(pd.DataFrame({
    "flag": ["is_high_conf", "high_confidence_anomaly_candidate"],
    "n_true": [int(df["is_high_conf"].sum()), int(df["high_confidence_anomaly_candidate"].sum())],
    "pct_true_%": [float(df["is_high_conf"].mean()*100), float(df["high_confidence_anomaly_candidate"].mean()*100)]
}))

# -----------------------------
# 2) Create within-slice percentiles for oe_ratio and residual
#    Slices: (HCPCS_Cd, Year) as our most comparable unit.
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    # percent rank in [0,1]; stable and easy
    return s.rank(pct=True, method="average")

df["oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["oe_ratio"].transform(_pct_rank)
df["resid_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["residual"].transform(_pct_rank)

# Convenience: top-x% flags
df["is_top_1pct_oe_in_slice"] = df["oe_pct_in_slice"] >= 0.99
df["is_top_1pct_resid_in_slice"] = df["resid_pct_in_slice"] >= 0.99

print("Percentile features created:", ["oe_pct_in_slice", "resid_pct_in_slice"])
display(df[["HCPCS_Cd","Year","oe_ratio","oe_pct_in_slice","residual","resid_pct_in_slice","services","benes","is_high_conf"]].head(5))

# Save back to the notebook namespace
eval_anom = df
print("Defined: eval_anom (copy of eval_scored_DG_V3 + anomaly features)")

# Quick sanity chech: Confirming the grains of `eval_scored_DG_V3` and `eval_anom` 

In [ ]:
keys = ["Rndrng_NPI","HCPCS_Cd","Place_Of_Srvc","Year"]
print("rows:", len(eval_scored_DG_V3))
print("unique key rows:", eval_scored_DG_V3[keys].drop_duplicates().shape[0])
print("max duplicates per key:", eval_scored_DG_V3.groupby(keys).size().max())

In [ ]:
keys = ["Rndrng_NPI","HCPCS_Cd","Place_Of_Srvc","Year"]
print("rows:", len(eval_anom))
print("unique key rows:", eval_anom[keys].drop_duplicates().shape[0])
print("max duplicates per key:", eval_anom.groupby(keys).size().max())

> **CONFIRMED**: `eval_scored_DG_V3` grain is exactly `(Rndrng_NPI, HCPCS_Cd, Place_Of_Srvc, Year)`. And, `eval_anom` has the exact same grain (we did not introduce any duplication through merges/feature adds).

# ANOM.2 Row-level “Top anomalies” action list

This is our “what should a stakeholder look at first” table

### ANOM.2) Row-level top anomalies (action list) (uses `oe_ratio` for slice ranking)

A row “pops” when it is over-expected and extreme within its peer slice:
- `oe_ratio` > 1.0 (over-expected)
- `residual` > 0 (over-expected in absolute dollars too)
- ranking signal uses within-slice percentiles on (HCPCS_Cd, Year):
    - `oe_pct_in_slice` (tail of O/E within slice)
    - `resid_pct_in_slice` (tail of residual within slice)
- score: `anom_score = 0.6*oe_pct_in_slice + 0.4*resid_pct_in_slice + 0.2*is_high_conf`
- rows are then ranked and we take top N (e.g., 200)

So here “extreme event” means:

High percentile tail on `oe_ratio` (and `residual`) within the HCPCS-year slice, conditional on `oe_ratio` > 1 and `residual` > 0.

In [ ]:
# ============================================================
# ANOM.2) Row-level top anomalies (action list)
# ============================================================

df = eval_anom.copy()

# Scope: focus on over-expected (positive residual or high OE)
# You can tighten/loosen these later.
row_candidates = df[
    (df["oe_ratio"].notna())
    & (df["expected_cost"].notna())
    & (df["observed_cost"].notna())
].copy()

# A simple, explainable scoring:
# - prioritize high OE percentile and residual percentile
# - boost if high-confidence
row_candidates["anom_score"] = (
    0.6 * row_candidates["oe_pct_in_slice"].fillna(0)
    + 0.4 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.2 * row_candidates["is_high_conf"].astype(int)
)

# Keep only over-expected direction (optional but usually desired)
row_candidates = row_candidates[(row_candidates["oe_ratio"] > 1.0) & (row_candidates["residual"] > 0)]

TOP_N = 200

top_anomalies_rows = (
    row_candidates.sort_values(["anom_score","oe_ratio","residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio",
        "oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
    ]]
)

print("Top anomalies (row-level):", len(top_anomalies_rows))
display(top_anomalies_rows.head(20))

# Save
anom_top_rows = top_anomalies_rows

### Checking the row numbers where `expected_cost` is 0 or ≤1. 

This will tell us whether we should specially handle the cases where `expected_cost` is very small leading to explosion in O/E ratio and creating false-positive anomaly signals. 

In [ ]:
print(f"eval_scored_DG_V3 has {eval_scored_DG_V3.expected_cost.eq(0).sum()} of rows with expected cost of 0 (zero) (out of {eval_scored_DG_V3.shape[0]} total rows)")
print(f"eval_scored_DG_V3 has {top_anomalies_rows.expected_cost.eq(0).sum()} of rows with expected cost of 0 (zero) (out of {top_anomalies_rows.shape[0]} total rows)")

> We don't need to create a special flag for `expected_cost == 0` because there are so few of them. However, we should implement using `log_oe_ratio` which is more robust than `oe_ratio` as it would compress the values for both `observed_cost` and `expected_cost` preventing explosive O/E ratios. 

# ANOM.2.a Row-level top anomalies (robust using `log_oe`)

### ANOM.2.a) Row-level top anomalies (action list) (uses ROBUST `log_oe_ratio` for slice ranking)

> Howeer, since we're still using `_pct_rank` (percentile rank) within each slice (i.e., `(HCPCS_Cd, Year)`) and not "magnitude-aware" approach, where the difference between using `oe_ratio` and `log_oe_ratio` would surface, the expected result is the same as "ANOM.2" above.

Same structure as ANOM.2, but the tail metric uses `log_oe` instead of raw `oe_ratio`:

- `log_oe` > 0 (over-expected on log scale, equivalent direction to `oe_ratio` > 1 when computed consistently)
- `residual` > 0
- slice percentile uses `log_oe_pct_in_slice` within `(HCPCS_Cd, Year)`
- score: `anom_score_robust = 0.6*log_oe_pct_in_slice + 0.4*resid_pct_in_slice + 0.2*is_high_conf`
- optional denominator hygiene: `expected_cost >= MIN_EXPECTED` (we used 1e-2)

So here “extreme event” means:

Top tail of log(O/E) within the HCPCS-year slice (plus residual tail), conditional on `log_oe` > 0 and `residual` > 0 (and optionally `expected_cost` not tiny).

In [ ]:
# ============================================================
# ANOM.2.a) Row-level top anomalies (ROBUST: uses log_oe)
# Minimal edit of ANOM.2:
#   - uses log_oe percentiles instead of oe_ratio percentiles
#   - filters on log_oe > 0 (equivalent to oe_ratio > 1)
#   - keeps residual > 0
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# -----------------------------
# 0) Build within-slice percentile for log_oe (HCPCS_Cd, Year)
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

if "log_oe" not in df.columns:
    raise KeyError("Expected column 'log_oe' not found in eval_anom. (It exists in eval_scored_DG_V3 schema we shared.)")

df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(_pct_rank)

# -----------------------------
# 1) Candidate universe (same idea as ANOM.2)
# -----------------------------
row_candidates = df[
    (df["log_oe"].notna())
    & (df["expected_cost"].notna())
    & (df["observed_cost"].notna())
    & (df["residual"].notna())
].copy()

# Optional denominator hygiene (recommended toggle)
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2  # 0.01; small but avoids ultra-tiny denominators driving extremes

if USE_MIN_EXPECTED_FILTER:
    row_candidates = row_candidates[row_candidates["expected_cost"] >= MIN_EXPECTED]

# -----------------------------
# 2) Scoring (log-based)
# -----------------------------
row_candidates["anom_score_robust"] = (
    0.6 * row_candidates["log_oe_pct_in_slice"].fillna(0)
    + 0.4 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.2 * row_candidates["is_high_conf"].astype(int)
)

# -----------------------------
# 3) Directional filter (robust analog of oe_ratio > 1 and residual > 0)
# -----------------------------
# log_oe > 0 <=> oe_ratio > 1.0
LOG_OE_MIN = 0.0

row_candidates = row_candidates[(row_candidates["log_oe"] > LOG_OE_MIN) & (row_candidates["residual"] > 0)]

TOP_N = 200

top_anomalies_rows_robust = (
    row_candidates.sort_values(["anom_score_robust", "log_oe", "residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
        "anom_score_robust",
    ]]
)

print("Top anomalies (row-level, robust log_oe):", len(top_anomalies_rows_robust))
display(top_anomalies_rows_robust.head(20))

anom_top_rows_robust = top_anomalies_rows_robust

# ANOM.2.a.1) Row-level top anomalies (ROBUST + size-aware: log_oe + slice_n filter)

### ANOM.2.a.1) Row-level top anomalies (ROBUST + size-aware: log_oe + slice_n filter) (filters out small slices after transforming with `_pct_rank()`)

> This builds on "ANOM.2.a" adding an extra robustness step with excluding small-size groups. 

Same as ANOM.2.a, plus slice-size validity:
- `slice_n` = `size(HCPCS_Cd, Year)`
- require `slice_n` >= 50
- keep: `log_oe` > 0, `residual` > 0, `log_oe_pct_in_slice`-based scoring, confidence boost, then top N

So here “extreme event” means:

Top tail of log(O/E) within sufficiently large HCPCS-year slices (n≥50), conditional on positive residual (and optional minimum expected).

In [ ]:
# ============================================================
# ANOM.2.a.1) Row-level top anomalies (ROBUST + size-aware: log_oe + slice_n filter)
# Minimal edit of ANOM.2.a:
#   - add slice_n per (HCPCS_Cd, Year)
#   - filter to slice_n >= MIN_SLICE_N
#   - include slice_n in output
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# -----------------------------
# 0) Build within-slice percentile for log_oe (HCPCS_Cd, Year)
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

if "log_oe" not in df.columns:
    raise KeyError("Expected column 'log_oe' not found in eval_anom.")

df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(_pct_rank)

# NEW: slice size awareness
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")
MIN_SLICE_N = 50

# -----------------------------
# 1) Candidate universe
# -----------------------------
row_candidates = df[
    (df["log_oe"].notna())
    & (df["expected_cost"].notna())
    & (df["observed_cost"].notna())
    & (df["residual"].notna())
].copy()

# NEW: filter out tiny slices
row_candidates = row_candidates[row_candidates["slice_n"] >= MIN_SLICE_N]

# Optional denominator hygiene (recommended toggle)
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2

if USE_MIN_EXPECTED_FILTER:
    row_candidates = row_candidates[row_candidates["expected_cost"] >= MIN_EXPECTED]

# -----------------------------
# 2) Scoring (log-based)
# -----------------------------
row_candidates["anom_score_robust"] = (
    0.6 * row_candidates["log_oe_pct_in_slice"].fillna(0)
    + 0.4 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.2 * row_candidates["is_high_conf"].astype(int)
)

row_candidates["anom_reason"] = "top_1pct_log_oe_in_slice + high_conf + positive_residual"

row_candidates["slice_key"] = (
    row_candidates["HCPCS_Cd"].astype(str) + "_" + row_candidates["Year"].astype(int).astype(str)
)

# -----------------------------
# 3) Directional filter
# -----------------------------
LOG_OE_MIN = 0.0
row_candidates = row_candidates[(row_candidates["log_oe"] > LOG_OE_MIN) & (row_candidates["residual"] > 0)]

TOP_N = 200

top_anomalies_rows_robust_sizeaware = (
    row_candidates.sort_values(["anom_score_robust", "log_oe", "residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "slice_n",  # NEW
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
        "anom_score_robust",
        "anom_reason",
        "slice_key"
    ]]
)

print("Top anomalies (row-level (within (HCPCS_Cd, Year) slice), robust + size-aware):", len(top_anomalies_rows_robust_sizeaware))
print(f"Applied slice size filter: slice_n >= {MIN_SLICE_N}")
display(top_anomalies_rows_robust_sizeaware.head(20))

anom_top_rows_robust_sizeaware = top_anomalies_rows_robust_sizeaware

# ANOM.2.b) Row-level top anomalies (magnitude-aware scoring; log_oe matters)

### ANOM.2.b) Row-level top anomalies (magnitude-aware scoring; log_oe matters) (computes a magnitude-based scoring so log-transforming `oe_ratio` is meaningful)

> Here we still compute the percentile of `log_oe` within each slice (i.e., within each `(HCPCS_Cd, Year)` silce) but then instead of taking the top ranking 1% from each slice, we clip `log_mag` to stabilize scoring and then score the providers taking `log_mag` into consideration.

A row “pops” when it is over-expected, passes denominator hygiene, and then ranks highly by a score that mixes slice-relative extremeness with absolute severity:

Row must satisfy (hard gates):
- `expected_cost` >= 1e-2 (denominator hygiene)
- `log_oe` > 0 (over-expected on log scale)
- `residual` > 0 (over-expected in absolute dollars)

Slice context (where “peer slice” = (HCPCS_Cd, Year)):
- `log_oe_pct_in_slice` = percentile rank of `log_oe` within `(HCPCS_Cd, Year)`
- `resid_pct_in_slice` = percentile rank of `residual` within `(HCPCS_Cd, Year)`

Magnitude severity term (absolute “how extreme”):
- `log_mag` = clip(log_oe, 0, 2.0) / 2.0
    - so anything above `log_oe` = 2.0 (about `oe_ratio` ≈ 7.39x) is capped for scoring stability

Score (used for ranking, not gating):
- `anom_score_mag` = 0.45*log_oe_pct_in_slice + 0.35*resid_pct_in_slice + 0.35*log_mag + 0.15*is_high_conf

Then we take Top N (200) by `anom_score_mag` (tie-breakers: higher log_oe, higher residual).

So here “extreme event” means:

Over-expected rows (`log_oe` > 0, `residual` > 0) with non-tiny expected cost, ranked by a blend of (a) top-tail position within the HCPCS-year peer slice and (b) absolute log(OE) severity (capped at `log_oe` = 2.0), with a confidence bump.

In [ ]:
# ============================================================
# ANOM.2.b) Row-level top anomalies (magnitude-aware scoring; log_oe matters)
# - keeps slice percentiles (comparability)
# - adds clipped log magnitude (severity)
# - optional expected_cost floor (denominator hygiene)
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# Preconditions
needed = ["log_oe", "resid_pct_in_slice", "is_high_conf", "expected_cost", "observed_cost", "residual", "HCPCS_Cd", "Year"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns for ANOM.2.b: {missing}. Run ANOM.1 first.")

# If log_oe_pct_in_slice isn't there, compute it (same slice as before)
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# Candidate universe
row_candidates = df[
    df["log_oe"].notna()
    & df["expected_cost"].notna()
    & df["observed_cost"].notna()
    & df["residual"].notna()
].copy()

# Denominator hygiene (recommended)
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2  # 0.01. Tune later based on expected_cost distribution.

if USE_MIN_EXPECTED_FILTER:
    row_candidates = row_candidates[row_candidates["expected_cost"] >= MIN_EXPECTED]

# Direction filter (over-expected)
row_candidates = row_candidates[(row_candidates["log_oe"] > 0) & (row_candidates["residual"] > 0)]

# Magnitude transform: clip log_oe so outliers do not dominate too hard.
# Interpretation:
#   log_oe = log(oe_ratio)
#   log_oe=0.0 => oe_ratio=1.0
#   log_oe~0.405 => oe_ratio~1.5
#   log_oe~0.693 => oe_ratio~2.0
#   log_oe~1.609 => oe_ratio~5.0
LOG_CLIP_MAX = 2.0   # oe_ratio ~ 7.39. Any bigger gets capped for scoring stability.
log_mag = np.clip(row_candidates["log_oe"].to_numpy(dtype="float64"), 0.0, LOG_CLIP_MAX) / LOG_CLIP_MAX

# Score: mix comparability + severity + confidence
# - 0.45: within-slice extremeness by log percentile
# - 0.35: within-slice extremeness by residual percentile
# - 0.35: magnitude severity by clipped log
# - 0.15: confidence bump
# Note: weights can exceed 1, that is fine. We care about ranking.
row_candidates["anom_score_mag"] = (
    0.45 * row_candidates["log_oe_pct_in_slice"].fillna(0)
    + 0.35 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.35 * log_mag
    + 0.15 * row_candidates["is_high_conf"].astype(int)
)

TOP_N = 200

top_anomalies_rows_mag = (
    row_candidates.sort_values(["anom_score_mag", "log_oe", "residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
        "anom_score_mag",
    ]]
)

print("Top anomalies (row-level (within (HCPCS_Cd, Year) slice), magnitude-aware):", len(top_anomalies_rows_mag))
display(top_anomalies_rows_mag.head(20))

anom_top_rows_mag = top_anomalies_rows_mag

# ANOM.2.b.1) Row-level top anomalies (magnitude-aware + size-aware)

This is ANOM.2.b plus “top-1% is only meaningful when the slice is big enough.”

Row must satisfy the same gates as 2.b, plus:
- `slice_n` = `size(HCPCS_Cd, Year)`
- `slice_n` >= 50

Everything else is identical:
- same hygiene: `expected_cost` >= 1e-2
- same direction: `log_oe` > 0 and `residual` > 0
- same severity: `log_mag` = `clip(log_oe, 0, 2.0)/2.0`
- same ranking score:
`anom_score_mag = 0.45*log_oe_pct_in_slice + 0.35*resid_pct_in_slice + 0.35*log_mag + 0.15*is_high_conf`
- then Top N (200)

So here “extreme event” means:

Over-expected, non-tiny-expected rows that are extreme within sufficiently large HCPCS-year slices (n≥50), and are additionally prioritized by absolute log(OE) severity (capped), not just slice percentile rank.

In [ ]:
# ============================================================
# ANOM.2.b.1) Row-level top anomalies (magnitude-aware + size-aware)
# Minimal edit of ANOM.2.b:
#   - add slice_n per (HCPCS_Cd, Year)
#   - filter to slice_n >= MIN_SLICE_N
#   - include slice_n in output
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

needed = ["log_oe", "resid_pct_in_slice", "is_high_conf", "expected_cost", "observed_cost", "residual", "HCPCS_Cd", "Year", "row_id"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns for ANOM.2.b.1: {missing}. Run ANOM.1 first.")

# If log_oe_pct_in_slice isn't there, compute it
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# NEW: slice size awareness
slice_cols = ["HCPCS_Cd", "Year"]
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")
MIN_SLICE_N = 50

row_candidates = df[
    df["log_oe"].notna()
    & df["expected_cost"].notna()
    & df["observed_cost"].notna()
    & df["residual"].notna()
].copy()

# NEW: filter out tiny slices
row_candidates = row_candidates[row_candidates["slice_n"] >= MIN_SLICE_N]

# Denominator hygiene
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2
if USE_MIN_EXPECTED_FILTER:
    row_candidates = row_candidates[row_candidates["expected_cost"] >= MIN_EXPECTED]

# Direction filter
row_candidates = row_candidates[(row_candidates["log_oe"] > 0) & (row_candidates["residual"] > 0)]

# Magnitude transform
LOG_CLIP_MAX = 2.0
log_mag = np.clip(row_candidates["log_oe"].to_numpy(dtype="float64"), 0.0, LOG_CLIP_MAX) / LOG_CLIP_MAX

row_candidates["anom_score_mag"] = (
    0.45 * row_candidates["log_oe_pct_in_slice"].fillna(0)
    + 0.35 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.35 * log_mag
    + 0.15 * row_candidates["is_high_conf"].astype(int)
)

row_candidates["anom_reason"] = "top_1pct_log_oe_in_slice + high_conf + positive_residual"

row_candidates["slice_key"] = (
    row_candidates["HCPCS_Cd"].astype(str) + "_" + row_candidates["Year"].astype(int).astype(str)
)

TOP_N = 200

top_anomalies_rows_mag_sizeaware = (
    row_candidates.sort_values(["anom_score_mag", "log_oe", "residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "slice_n",  # NEW
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
        "anom_score_mag",
        "anom_reason",
        "slice_key"
    ]]
)

print("Top anomalies (row-level (within (HCPCS_Cd, Year) slice), magnitude-aware + size-aware):", len(top_anomalies_rows_mag_sizeaware))
print(f"Applied slice size filter: slice_n >= {MIN_SLICE_N}")
display(top_anomalies_rows_mag_sizeaware.head(20))

anom_top_rows_mag_sizeaware = top_anomalies_rows_mag_sizeaware

# Summary of row-level anomaly analyses ANOM.2. ANOM.2.a, ANOM.2.a.1, ANOM.2.b, and ANOM.2.b.1.

# ✅ Row-level anomaly surfacing variants (ANOM.2 family)

This section documents the five row-level anomaly approaches we implemented, why each exists, what they share, and what makes each one different. The goal is to have a clear, auditable “two-product” workflow:

Primary product (recommended): ANOM.2.a.1 (log-based + slice-size aware)  
Alternative product (contrast / “magnitude-first”): ANOM.2.b.1 (magnitude-aware + slice-size aware)

## 🎯 What “row-level anomalies” mean here

A row is one provider-service-year observation (roughly: NPI × HCPCS × Year, plus context). A row is suspicious in the over-expected direction when:

- observed cost is above expected, and
- the row looks extreme within its peer slice, and
- we have enough support to trust it (confidence rules), and
- (in size-aware variants) the peer slice is large enough to make “top 1%” meaningful.

## ✅ Common building blocks across all variants

### 1) Candidate universe (basic validity)

All approaches start by filtering to rows with valid core fields:

- observed_cost not null
- expected_cost not null
- residual not null (for robust variants)
- OE or log(OE) not null

### 2) Directionality (over-expected only)

Every approach keeps the anomaly direction consistent:

- positive residual (residual > 0)
- and over-expected:
  - non-log versions: oe_ratio > 1
  - log versions: log_oe > 0 (equivalent to oe_ratio > 1)

### 3) Confidence boost

All approaches incorporate “confidence” so the list is actionable, not noise:

- is_high_conf (services + benes + support tier) enters as a score boost
- we also keep high_confidence_anomaly_candidate in the output table as a fast-track flag for review

### 4) Slice-comparable ranking

All variants rely on “comparability slices”:

- slice key is (HCPCS_Cd, Year) by default
- percentiles are computed within each slice to avoid comparing unrelated services

## 🧠 What differs between variants

We have two main design choices:

### A) How do we measure “over-expected extremeness”?

- ANOM.2: uses oe_ratio percentile inside slice (oe_pct_in_slice)
- ANOM.2.a: uses log_oe percentile inside slice (log_oe_pct_in_slice)
- ANOM.2.b: uses a magnitude-aware score so very large log_oe values get extra emphasis (not just rank)

### B) Do we treat tiny slices as trustworthy?

- Non size-aware: top 1% can be misleading if slice_n is small
- Size-aware (.1 variants): compute slice_n and require slice_n >= 50

## 🧾 Summary of each approach

### ANOM.2 (baseline percentile approach using OE)

Builds within-slice percentiles for:

- oe_ratio (oe_pct_in_slice)
- residual (resid_pct_in_slice)

Scores rows using a weighted percentile blend + confidence boost:

`anom_score = 0.6*oe_pct + 0.4*resid_pct + 0.2*is_high_conf`

Filters:

- oe_ratio > 1 and residual > 0

✅ Good: simple and intuitive  
⚠️ Weakness: OE explodes when expected is tiny (even if rare), and OE tails are skewed

### ANOM.2.a (robust percentile approach using log(OE))

Adds within-slice percentile for:

- log_oe (log_oe_pct_in_slice)

Uses percentiles (still rank-based), but with a more stable signal than raw OE:

`anom_score_robust = 0.6*log_oe_pct + 0.4*resid_pct + 0.2*is_high_conf`

Filters:

- log_oe > 0 and residual > 0

Optional denominator hygiene:

- expected_cost >= MIN_EXPECTED (default 1e-2)

✅ Good: keeps the percentile logic but reduces distortion from extreme OE tails  
⚠️ Weakness: still percentile-based, so “top 1%” depends on slice size

### ANOM.2.a.1 (robust + size-aware) ✅ Recommended

This is 2.a plus slice-size awareness.

Computes:

- slice_n = groupby(HCPCS_Cd, Year).size

Adds filter:

- slice_n >= 50

Keeps:

- log_oe_pct_in_slice, residual percentile, confidence boost

✅ Best for actionability: avoids “winner of a tiny slice” problem  
✅ Most defensible: auditable, stable, comparable, and size-aware  
⚠️ Slight tradeoff: we may miss rare codes with small slices, but that’s a feature not a bug

### ANOM.2.b (magnitude-aware scoring)

This variant is intentionally different. It tries to make ranking sensitive to how extreme the anomaly is, not only whether it’s in the top 1%.

Uses:

- log_oe magnitude (and possibly residual magnitude)

Still keeps percentiles as context, but the score is magnitude-driven.  
Output tends to concentrate around codes with many extremely high log_oe rows (like J3490).

✅ Good: produces a “shock list” of the most extreme rows  
⚠️ Weakness: can over-focus on a few codes where expected is systematically low or unstable, even if slice-relative percentile is already maxed out

### ANOM.2.b.1 (magnitude-aware + size-aware) ✅ Alternative product

This is 2.b plus slice-size awareness.

Adds:

- slice_n >= 50

Keeps:

- magnitude-aware scoring focus

✅ Great “alternative product” to contrast with 2.a.1  
✅ More interpretable for stakeholders who care about absolute extremeness  
⚠️ Can still over-index on a single HCPCS family if it dominates magnitude tails

## 📊 Comparison table (granular)

| Variant | Core OE signal used | “Extremeness” definition (explicit) | Score type | Directional filter (hard gate) | Denominator hygiene (hard gate) | Slice-size awareness (hard gate) | Strengths | Risks / failure modes | Best use |
|---|---|---|---|---|---|---|---|---|---|
| ANOM.2 | oe_ratio | **Slice-relative tail** within slice = (HCPCS_Cd, Year): uses `oe_pct_in_slice` (pct-rank of `oe_ratio`) plus `resid_pct_in_slice` (pct-rank of `residual`) | Percentile-weighted (`0.6*oe_pct + 0.4*resid_pct + 0.2*is_high_conf`) | `oe_ratio > 1` and `residual > 0` | None | No | Simple, intuitive baseline | Raw OE tails can be distorted when expected is small | First baseline, sanity check |
| ANOM.2.a | log_oe | **Slice-relative tail** within slice = (HCPCS_Cd, Year): uses `log_oe_pct_in_slice` (pct-rank of `log_oe`) plus `resid_pct_in_slice` | Percentile-weighted (`0.6*log_oe_pct + 0.4*resid_pct + 0.2*is_high_conf`) | `log_oe > 0` and `residual > 0` | Optional `expected_cost >= MIN_EXPECTED` | No | More stable than raw OE, still slice-comparable | “Top 1%” interpretation weak for tiny slices | Main method before size filter |
| ANOM.2.a.1 ✅ | log_oe | **Slice-relative tail + slice validity**: `log_oe_pct_in_slice` + `resid_pct_in_slice`, both within (HCPCS_Cd, Year), but only when the slice is big enough | Percentile-weighted (`0.6*log_oe_pct + 0.4*resid_pct + 0.2*is_high_conf`) | `log_oe > 0` and `residual > 0` | Optional `expected_cost >= MIN_EXPECTED` | ✅ `slice_n >= 50` | Most defensible, avoids tiny-slice “wins” | Intentionally drops rare-code tiny-slice anomalies | Primary action list |
| ANOM.2.b | log_oe (magnitude) | **Slice-relative tail + magnitude severity**: keeps `log_oe_pct_in_slice` and `resid_pct_in_slice` (within (HCPCS_Cd, Year)) *and* adds a capped magnitude term `log_mag = clip(log_oe,0,LOG_CLIP_MAX)/LOG_CLIP_MAX` | Magnitude-aware (`0.45*log_oe_pct + 0.35*resid_pct + 0.35*log_mag + 0.15*is_high_conf`) | `log_oe > 0` and `residual > 0` | Usually `expected_cost >= MIN_EXPECTED` | No | Finds biggest “shocks” (severity-forward) | Can over-focus on a few HCPCS families that dominate magnitude tails | Alternative shock list |
| ANOM.2.b.1 ✅ | log_oe (magnitude) | **Slice-relative tail + magnitude severity + slice validity**: same as 2.b, but only for sufficiently large (HCPCS_Cd, Year) slices | Magnitude-aware (`0.45*log_oe_pct + 0.35*resid_pct + 0.35*log_mag + 0.15*is_high_conf`) | `log_oe > 0` and `residual > 0` | Usually `expected_cost >= MIN_EXPECTED` | ✅ `slice_n >= 50` | Shock list that is also defensible | Still can be dominated by a few HCPCS codes, but less “tiny-slice noise” | Secondary product |
## 🧩 “Two-product” framing

### ✅ Product 1:  
ANOM.2.a.1 (Primary, robust + size-aware)

- Best for recurring use
- Least likely to waste reviewer time
- More stable across reruns and code distribution changes

### ✅ Product 2:  
ANOM.2.b.1 (Alternative, magnitude-first + size-aware)

- Best for “what are the most extreme dollar shocks?”
- Useful as a contrast view
- Helps us catch “big outliers” that might not rank highest purely by percentile logic

# Quick check: keep the `MIN_EXPECTED` hygiene toggle enabled our robust row-level variants

> This table will tell us exactly how aggressive our denominator hygiene needs to be, and it also explains why our “robust” variants mostly looked fine even before we tuned `MIN_EXPECTED`.

In [ ]:
tolerances = [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
counts_eval_scored_DG_V3_obs = []
counts_eval_scored_DG_V3_exp = []
counts_row_candidates_obs = []
counts_row_candidates_exp = []

for tol in tolerances:
    counts_eval_scored_DG_V3_obs.append((eval_scored_DG_V3["observed_cost"].abs() < tol).sum())
    counts_eval_scored_DG_V3_exp.append((eval_scored_DG_V3["expected_cost"].abs() < tol).sum())
    counts_row_candidates_obs.append((row_candidates["observed_cost"].abs() < tol).sum())
    counts_row_candidates_exp.append((row_candidates["expected_cost"].abs() < tol).sum())

c = ["< 1", "< 1e-1", "< 1e-2", "< 1e-3", "< 1e-4", "< 1e-5", "< 1e-6"]

temp = pd.DataFrame(
    [
        counts_eval_scored_DG_V3_obs,
        counts_eval_scored_DG_V3_exp,
        counts_row_candidates_obs,
        counts_row_candidates_exp
    ],
    columns=c
)

temp.index = ["eval_scored_DG_V3 observed", "eval_scored_DG_V3 expected", "row_candidates observed", "row_candidates expected"]

temp.reset_index().rename(columns={"index":"source"})

**What the counts say**

1) In the full eval universe (`eval_scored_DG_V3`)

- `expected_cost` < 1 happens a lot: 82,079 rows.
- `expected_cost` < 0.1 is still non-trivial: 11,180 rows.
- Then it collapses ***fast***:
    - 0.01: 108
    - 0.001: 41
    - 1e-4: 39
    - 1e-6: 39 (these are basically the “effectively zero expected” rows)

So the true “OE explosion” landmine is mostly those 39 rows (and to a lesser extent the 41/108 rows).

2) In `row_candidates`

We’re seeing something important:
- `row_candidates` `expected_cost` < 0.01 = 108
- `row_candidates` `expected_cost` < 1e-6 = 39

That means our `row_candidates` construction is currently letting all the “tiny expected” rows through (unless we filter them later).

Also:
- `row_candidates` `observed_cost` < 0.01 = 6 and < 0.001 = 1
    - ***so most “tiny expected” rows are not tiny observed***. That’s exactly the setup that creates gigantic `oe_ratio`.

What this implies for `MIN_EXPECTED`

If our goal is “remove only true OE explosions”

Set:
- `MIN_EXPECTED = 1e-4` (or 1e-6)
- This removes only ~39 rows (basically “expected cost collapse” cases).
- Almost zero impact on real anomaly lists, but prevents pathological infinities and absurd ratios.

If our goal is “avoid unstable OE inflation from very small expected”

Set:
- `MIN_EXPECTED = 1e-2` (0.01) (our current default)
- Removes 108 rows. Still tiny, still safe.
- This is a good default because it cuts the “tiny denominator” corner cases without discarding meaningful low-cost rows.

If we crank it up to 1e-1 (0.1)
- We drop 5,830 rows in `row_candidates` and 11,180 rows in `eval_scored_DG_V3`.
- ***That is no longer “hygiene”. That’s a modeling/business decision.***
- We'll’ll start excluding lots of legitimate low-cost services and low-cost expected lines that might still be interesting (especially if residuals are meaningful).


# ANOM.3 Provider-level “who consistently pops” summary

This answers: “Which NPIs keep showing up across codes/years?”

Here, a provider “pops” when they have many rows where:
- `oe_ratio` > 1 (over-expected)
- `residual` > 0 (over-expected in absolute dollars too)
- `is_high_conf` OR `high_confidence_anomaly_candidate`
- `oe_pct_in_slice` >= 0.99 where the slice is `(HCPCS_Cd, Year)`

So here “extreme event” means:

Top 1% O/E within that HCPCS-year slice, conditional on positive residual and confidence.

In [ ]:
# ============================================================
# ANOM.3) Provider-level anomaly summary (who consistently pops)
# ============================================================

df = eval_anom.copy()

# Define "anomalous row" using our fast-track + percentile approach
df["is_row_anomalous"] = (
    (df["oe_ratio"] > 1.0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["oe_pct_in_slice"] >= 0.99)
)

provider_summary = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
    .agg(
        n_rows=("row_id","size"),
        n_anom_rows=("is_row_anomalous","sum"),
        anom_rate_pct=("is_row_anomalous", lambda s: float(s.mean()*100)),
        n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
        n_unique_years=("Year", pd.Series.nunique),
        median_oe=("oe_ratio","median"),
        median_residual=("residual","median"),
        total_services=("services","sum"),
        total_benes=("benes","sum"),
    )
    .reset_index()
)

# Rank: lots of anomalous rows + not just tiny footprint
provider_summary["provider_anom_score"] = (
    provider_summary["n_anom_rows"]
    + 0.25 * provider_summary["n_unique_codes"]
    + 0.25 * provider_summary["n_unique_years"]
)

TOP_N = 200
anom_top_providers = provider_summary.sort_values(
    ["provider_anom_score","n_anom_rows","anom_rate_pct","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers:", len(anom_top_providers))
print("Explanation: These providers accumulate the most row-level tail events, where a tail event is defined as a row with oe_pct_in_slice >= 0.99 computed within (HCPCS_Cd, Year).")
display(anom_top_providers.head(30))

# Save
anom_top_providers = anom_top_providers

# ANOM.3.a Provider-level anomaly summary (robust provider-level “who consistently pops”) (ROBUST: uses `log_oe`)

Same structure as ANOM.3, but the percentile tail is computed on `log_oe` instead of `oe_ratio`:
- `log_oe` > 0 (equivalent to `oe_ratio` > 1 if computed consistently)
- residual > 0
- `is_high_conf` OR `high_confidence_anomaly_candidate`
- `log_oe_pct_in_slice` >= 0.99 within `(HCPCS_Cd, Year)`

So “extreme event” means:

- Top 1% log(OE) within that HCPCS-year slice, conditional on residual and confidence.

In [ ]:
# ============================================================
# ANOM.3.a) Provider-level anomaly summary (ROBUST: uses log_oe)
# Self-contained: computes log_oe_pct_in_slice if missing
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# Preconditions that must exist already
for col in ["log_oe", "residual", "is_high_conf", "high_confidence_anomaly_candidate"]:
    if col not in df.columns:
        raise KeyError(f"Missing required column for ANOM.3.a: {col}.")

# If log_oe_pct_in_slice isn't persisted, compute it now
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]

    def _pct_rank(s: pd.Series) -> pd.Series:
        return s.rank(pct=True, method="average")

    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(_pct_rank)

# Robust anomalous row definition
df["is_row_anomalous_robust"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
)

provider_summary_robust = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
    .agg(
        n_rows=("row_id","size"),
        n_anom_rows_robust=("is_row_anomalous_robust","sum"),
        anom_rate_pct_robust=("is_row_anomalous_robust", lambda s: float(s.mean()*100)),
        n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
        n_unique_years=("Year", pd.Series.nunique),
        median_oe=("oe_ratio","median"),
        median_log_oe=("log_oe","median"),
        median_residual=("residual","median"),
        total_services=("services","sum"),
        total_benes=("benes","sum"),
    )
    .reset_index()
)

provider_summary_robust["provider_anom_score_robust"] = (
    provider_summary_robust["n_anom_rows_robust"]
    + 0.25 * provider_summary_robust["n_unique_codes"]
    + 0.25 * provider_summary_robust["n_unique_years"]
)

TOP_N = 200
anom_top_providers_robust = provider_summary_robust.sort_values(
    ["provider_anom_score_robust","n_anom_rows_robust","anom_rate_pct_robust","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers (robust log_oe):", len(anom_top_providers_robust))
display(anom_top_providers_robust.head(30))

anom_top_providers_robust = anom_top_providers_robust

# ANOM.3.a.1) Provider-level anomaly summary (ROBUST + size-aware: log_oe + slice_n)

Same as ANOM.3.a, plus:
- `slice_n` >= 50 for the slice `(HCPCS_Cd, Year)`

So “extreme event” means:

Top 1% log(OE) within sufficiently large HCPCS-year slices (n≥50), conditional on residual and confidence.

In [ ]:
# ============================================================
# ANOM.3.a.1) Provider-level anomaly summary (ROBUST + size-aware: log_oe + slice_n)
# Minimal edit of ANOM.3.a:
#   - add slice_n per (HCPCS_Cd, Year)
#   - require slice_n >= MIN_SLICE_N for anomalous-row definition
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

for col in ["log_oe", "residual", "is_high_conf", "high_confidence_anomaly_candidate", "HCPCS_Cd", "Year", "row_id"]:
    if col not in df.columns:
        raise KeyError(f"Missing required column for ANOM.3.a.1: {col}.")

# Ensure log_oe_pct_in_slice exists
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# NEW: slice size awareness
slice_cols = ["HCPCS_Cd", "Year"]
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")
MIN_SLICE_N = 50

df["is_row_anomalous_robust_sizeaware"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["slice_n"] >= MIN_SLICE_N)  # NEW
)

provider_summary_robust_sizeaware = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
    .agg(
        n_rows=("row_id","size"),
        n_anom_rows_robust=("is_row_anomalous_robust_sizeaware","sum"),
        anom_rate_pct_robust=("is_row_anomalous_robust_sizeaware", lambda s: float(s.mean()*100)),
        n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
        n_unique_years=("Year", pd.Series.nunique),
        median_oe=("oe_ratio","median"),
        median_log_oe=("log_oe","median"),
        median_residual=("residual","median"),
        total_services=("services","sum") if "services" in df.columns else ("row_id","size"),
        total_benes=("benes","sum") if "benes" in df.columns else ("row_id","size"),
    )
    .reset_index()
)

provider_summary_robust_sizeaware = provider_summary_robust_sizeaware[
    provider_summary_robust_sizeaware["n_anom_rows_robust"] > 0
].copy()

provider_summary_robust_sizeaware["provider_anom_score_robust"] = (
    provider_summary_robust_sizeaware["n_anom_rows_robust"]
    + 0.25 * provider_summary_robust_sizeaware["n_unique_codes"]
    + 0.25 * provider_summary_robust_sizeaware["n_unique_years"]
)


TOP_N = 200
anom_top_providers_robust_sizeaware = provider_summary_robust_sizeaware.sort_values(
    ["provider_anom_score_robust","n_anom_rows_robust","anom_rate_pct_robust","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers (robust + size-aware):", len(anom_top_providers_robust_sizeaware))
print(f"Applied slice size filter in row definition: slice_n >= {MIN_SLICE_N}")
display(anom_top_providers_robust_sizeaware.head(30))

anom_top_providers_robust_sizeaware = anom_top_providers_robust_sizeaware

# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# Persist the flag `is_row_anomalous_robust_sizeaware` into `eval_anom` for downstream viz

eval_anom = eval_anom.merge(
    df[["row_id", "is_row_anomalous_robust_sizeaware"]],
    on="row_id",
    how="left"
)
eval_anom["is_row_anomalous_robust_sizeaware"] = eval_anom["is_row_anomalous_robust_sizeaware"].fillna(False)

# ANOM.3.b) Provider-level anomaly summary (magnitude-aware; log_oe severity)

A provider “pops” when they have many rows where:
- `log_oe` > 0 (over-expected)
- `residual` > 0 (over-expected in absolute dollars too)
- `is_high_conf` OR `high_confidence_anomaly_candidate`
- `log_oe_pct_in_slice` >= 0.99 where the slice is `(HCPCS_Cd, Year)`
- `log_oe` >= `log_oe_severity_cut`, where `log_oe_severity_cut` is a global high-quantile cutoff computed from the over-expected base (we used `q=0.995`, about `oe_ratio` ~ 2.21x in our run)

So here “extreme event” means:

Top 1% within the HCPCS-year slice and “severe” in absolute terms globally (log_oe above the global severity threshold), conditional on positive residual and confidence.

In [ ]:
# ============================================================
# ANOM.3.b) Provider-level anomaly summary (magnitude-aware; log_oe severity)
# - defines is_row_anomalous_mag
# - ranks providers by count + breadth + severity
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# Preconditions
needed = ["log_oe", "residual", "is_high_conf", "high_confidence_anomaly_candidate", "HCPCS_Cd", "Year", "row_id", "Rndrng_NPI", "provider_type", "state"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns for ANOM.3.b: {missing}. Run ANOM.1 first.")

# Ensure log_oe_pct_in_slice exists
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# Optional denominator hygiene for the anomaly definition
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2
if USE_MIN_EXPECTED_FILTER and "expected_cost" in df.columns:
    df = df[df["expected_cost"].notna() & (df["expected_cost"] >= MIN_EXPECTED)].copy()

# Choose a magnitude threshold in a non-arbitrary way:
# - take a high quantile of log_oe among over-expected rows
# - this adapts to our data distribution
BASE = df[(df["log_oe"] > 0) & (df["residual"] > 0)]
if len(BASE) == 0:
    raise ValueError("No over-expected rows found (log_oe>0 & residual>0). Check inputs.")
LOG_OE_Q = 0.995  # top 0.5% severity threshold; tune for broader/narrower top slice
log_oe_severity_cut = float(BASE["log_oe"].quantile(LOG_OE_Q))

print(f"log_oe severity cutoff at q={LOG_OE_Q}: {log_oe_severity_cut:.4f} (oe_ratio ~ {np.exp(log_oe_severity_cut):.2f}x)")

# Magnitude-aware anomalous row:
# - direction: over-expected
# - confidence: high_conf OR fast-track flag
# - peer extremeness: top 1% within slice by log percentile
# - severity: log_oe above global severity cutoff
df["is_row_anomalous_mag"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["log_oe"] >= log_oe_severity_cut)
)

# provider_summary_mag = (
#     df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
#     .agg(
#         n_rows=("row_id","size"),
#         n_anom_rows_mag=("is_row_anomalous_mag","sum"),
#         anom_rate_pct_mag=("is_row_anomalous_mag", lambda s: float(s.mean()*100)),
#         n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
#         n_unique_years=("Year", pd.Series.nunique),

#         # Severity summaries among anomalous rows only
#         max_log_oe_mag=("log_oe", lambda s: float(np.nanmax(s))),
#         p95_log_oe_mag=("log_oe", lambda s: float(np.nanpercentile(s, 95))),
#         median_log_oe=("log_oe","median"),

#         total_services=("services","sum") if "services" in df.columns else ("row_id","size"),
#         total_benes=("benes","sum") if "benes" in df.columns else ("row_id","size"),
#     )
#     .reset_index()
# )

def _masked_max_log_oe(g):
    s = g.loc[g["is_row_anomalous_mag"], "log_oe"]
    return float(s.max()) if len(s) else np.nan

def _masked_p95_log_oe(g):
    s = g.loc[g["is_row_anomalous_mag"], "log_oe"]
    return float(np.nanpercentile(s, 95)) if len(s) else np.nan

provider_summary_mag = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "n_anom_rows_mag": int(g["is_row_anomalous_mag"].sum()),
          "anom_rate_pct_mag": float(g["is_row_anomalous_mag"].mean() * 100),
          "n_unique_codes": g["HCPCS_Cd"].nunique(),
          "n_unique_years": g["Year"].nunique(),
          "max_log_oe_mag": _masked_max_log_oe(g),
          "p95_log_oe_mag": _masked_p95_log_oe(g),
          "median_log_oe": float(g["log_oe"].median()),
          "total_services": float(g["services"].sum()),
          "total_benes": float(g["benes"].sum()),
      }), include_groups=False)
      .reset_index()
)


provider_summary_mag = provider_summary_mag.loc[
    provider_summary_mag["n_anom_rows_mag"] > 0
].copy()

# Rank providers by:
# - count of magnitude anomalies
# - breadth across codes/years
# - plus a small severity bump (p95_log_oe_mag)
provider_summary_mag["provider_anom_score_mag"] = (
    provider_summary_mag["n_anom_rows_mag"]
    + 0.25 * provider_summary_mag["n_unique_codes"]
    + 0.25 * provider_summary_mag["n_unique_years"]
    + 0.10 * provider_summary_mag["p95_log_oe_mag"]
)

TOP_N = 200
anom_top_providers_mag = provider_summary_mag.sort_values(
    ["provider_anom_score_mag","n_anom_rows_mag","anom_rate_pct_mag","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers (magnitude-aware):", len(anom_top_providers_mag))
display(anom_top_providers_mag.head(30))

anom_top_providers_mag = anom_top_providers_mag

# ANOM.3.b.1) Provider-level anomaly summary (magnitude-aware + size-aware)

Same as ANOM.3.b, plus:
- `slice_n` >= 50 for the slice `(HCPCS_Cd, Year)` (so “top 1%” is only trusted in adequately sized slices)

So here “extreme event” means:

Top 1% within sufficiently large HCPCS-year slices (n≥50) and “severe” globally (log_oe above the global severity cutoff), conditional on positive residual and confidence.

In [ ]:
# ============================================================
# ANOM.3.b.1) Provider-level anomaly summary (magnitude-aware + size-aware)
# Minimal edit of ANOM.3.b:
#   - add slice_n per (HCPCS_Cd, Year)
#   - require slice_n >= MIN_SLICE_N for anomalous-row definition
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

needed = ["log_oe", "residual", "is_high_conf", "high_confidence_anomaly_candidate", "HCPCS_Cd", "Year", "row_id", "Rndrng_NPI", "provider_type", "state"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns for ANOM.3.b.1: {missing}.")

# Ensure log_oe_pct_in_slice exists
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# NEW: slice size awareness
slice_cols = ["HCPCS_Cd", "Year"]
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")
MIN_SLICE_N = 50

# Optional denominator hygiene
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2
if USE_MIN_EXPECTED_FILTER and "expected_cost" in df.columns:
    df = df[df["expected_cost"].notna() & (df["expected_cost"] >= MIN_EXPECTED)].copy()

# Severity cutoff computed like in ANOM.3.b
BASE = df[(df["log_oe"] > 0) & (df["residual"] > 0)]
if len(BASE) == 0:
    raise ValueError("No over-expected rows found (log_oe>0 & residual>0). Check inputs.")
LOG_OE_Q = 0.995
log_oe_severity_cut = float(BASE["log_oe"].quantile(LOG_OE_Q))
print(f"log_oe_severity_cut (q={LOG_OE_Q}): {log_oe_severity_cut}")

print(f"log_oe severity cutoff at q={LOG_OE_Q}: {log_oe_severity_cut:.4f} (oe_ratio ~ {np.exp(log_oe_severity_cut):.2f}x)")

df["is_row_anomalous_mag_sizeaware"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["log_oe"] >= log_oe_severity_cut)
    & (df["slice_n"] >= MIN_SLICE_N)  # NEW
)

# provider_summary_mag_sizeaware = (
#     df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
#     .agg(
#         n_rows=("row_id","size"),
#         n_anom_rows_mag=("is_row_anomalous_mag_sizeaware","sum"),
#         anom_rate_pct_mag=("is_row_anomalous_mag_sizeaware", lambda s: float(s.mean()*100)),
#         n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
#         n_unique_years=("Year", pd.Series.nunique),
#         max_log_oe_mag=("log_oe", lambda s: float(np.nanmax(s))),
#         p95_log_oe_mag=("log_oe", lambda s: float(np.nanpercentile(s, 95))),
#         median_log_oe=("log_oe","median"),
#         total_services=("services","sum") if "services" in df.columns else ("row_id","size"),
#         total_benes=("benes","sum") if "benes" in df.columns else ("row_id","size"),
#     )
#     .reset_index()
# )

def _masked_max_log_oe(g):
    s = g.loc[g["is_row_anomalous_mag_sizeaware"], "log_oe"]
    return float(s.max()) if len(s) else np.nan

def _masked_p95_log_oe(g):
    s = g.loc[g["is_row_anomalous_mag_sizeaware"], "log_oe"]
    return float(np.nanpercentile(s, 95)) if len(s) else np.nan

provider_summary_mag_sizeaware = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "n_anom_rows_mag": int(g["is_row_anomalous_mag_sizeaware"].sum()),
          "anom_rate_pct_mag": float(g["is_row_anomalous_mag_sizeaware"].mean() * 100),
          "n_unique_codes": g["HCPCS_Cd"].nunique(),
          "n_unique_years": g["Year"].nunique(),
          "max_log_oe_mag": _masked_max_log_oe(g),
          "p95_log_oe_mag": _masked_p95_log_oe(g),
          "median_log_oe": float(g["log_oe"].median()),
          "total_services": float(g["services"].sum()),
          "total_benes": float(g["benes"].sum()),
      }), include_groups = False)
      .reset_index()
)

provider_summary_mag_sizeaware = provider_summary_mag_sizeaware.loc[
    provider_summary_mag_sizeaware["n_anom_rows_mag"] > 0
].copy()

provider_summary_mag_sizeaware["provider_anom_score_mag"] = (
    provider_summary_mag_sizeaware["n_anom_rows_mag"]
    + 0.25 * provider_summary_mag_sizeaware["n_unique_codes"]
    + 0.25 * provider_summary_mag_sizeaware["n_unique_years"]
    + 0.10 * provider_summary_mag_sizeaware["p95_log_oe_mag"]
)

TOP_N = 200
anom_top_providers_mag_sizeaware = provider_summary_mag_sizeaware.sort_values(
    ["provider_anom_score_mag","n_anom_rows_mag","anom_rate_pct_mag","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers (magnitude-aware + size-aware):", len(anom_top_providers_mag_sizeaware))
print(f"Applied slice size filter in row definition: slice_n >= {MIN_SLICE_N}")
display(anom_top_providers_mag_sizeaware.head(30))

anom_top_providers_mag_sizeaware = anom_top_providers_mag_sizeaware

# ✅ Provider-level anomaly surfacing variants (ANOM.3 family)

This section documents the five provider-level anomaly approaches we implemented, why each exists, what they share, and what makes each one different. The goal is to have a clear, auditable “two-product” workflow:
- ✅ Primary product (recommended): ANOM.3.a.1 (log-based + slice-size aware)
- ✅ Alternative product (contrast, “severity-first”): ANOM.3.b.1 (magnitude-aware + slice-size aware)

---

## 🎯 What “provider-level anomalies” mean here

A provider-level anomaly summary answers:

“Which providers repeatedly show up as extreme over-expected relative to comparable peers?”

Key idea: we are not ranking providers by average cost. We are ranking them by the count and breadth of row-level tail events that pass our rules.

A provider “consistently pops” when they have:
- multiple anomalous rows (NPI × HCPCS × Year rows),
- across multiple codes and years,
- in slices where the peer group is large enough to trust the percentile meaning (in size-aware variants).

---

## ✅ Common building blocks across all variants

### 1) Row-level anomaly flag drives everything

Every provider table begins by defining a boolean row flag (one row = one provider-service-year record). That row flag is then aggregated to provider-level counts and rates.

### 2) Directionality: over-expected only

All variants enforce the same “direction”:
- residual > 0 (observed above expected)
- and over-expected signal:
  - OE versions: oe_ratio > 1
  - log versions: log_oe > 0 (equivalent to OE > 1)

### 3) Confidence gating

All variants require rows to be credible using:
- is_high_conf (our auditable rule: support tier + minimum services + minimum benes)
- OR high_confidence_anomaly_candidate (our legacy fast-track flag)

### 4) Slice-comparable extremeness

All variants use the peer slice (HCPCS_Cd, Year) to avoid comparing unlike services.
- percentile computed inside slice, then used as “is this row extreme relative to peers for the same code/year?”

### 5) Provider ranking uses “count + breadth”

Every provider score is built around:
- how many anomalous rows (n_anom_rows_*)
- breadth across codes (n_unique_codes)
- breadth across years (n_unique_years)

---

## 🧠 What differs between variants

We have two main design choices:

### A) What is the “peer extremeness” signal?
- ANOM.3: oe_pct_in_slice (percentile of oe_ratio within slice)
- ANOM.3.a: log_oe_pct_in_slice (percentile of log_oe within slice, same ordering as OE but numerically better behaved)
- ANOM.3.b: adds a global severity cutoff log_oe >= quantile(...) so “how huge is it” matters

### B) Do we trust tiny slices?
- Non size-aware: “top 1%” can be meaningless when slice size is small
- Size-aware (.1 variants): compute slice_n and require slice_n >= 50

---

## 🧾 Summary of each approach

### ANOM.3 (baseline provider tail frequency using OE)

- Defines anomalous row:
  - oe_ratio > 1 and residual > 0
  - (is_high_conf OR high_confidence_anomaly_candidate)
  - oe_pct_in_slice >= 0.99
- Aggregates per provider:
  - counts (n_anom_rows), rates, breadth, medians, totals
- Ranks by:
  - n_anom_rows + 0.25*n_unique_codes + 0.25*n_unique_years

✅ Good: simple, intuitive baseline  
⚠️ Weakness: raw OE can be distorted by denominator issues, and “top 1%” can be misleading in tiny slices

---

### ANOM.3.a (robust provider tail frequency using log(OE))

- Same logic as ANOM.3 but uses:
  - log_oe > 0 instead of oe_ratio > 1
  - log_oe_pct_in_slice >= 0.99 instead of oe_pct_in_slice >= 0.99

✅ Good: same conceptual logic, less prone to OE numeric weirdness  
⚠️ Important: because log is monotonic, the top-1% set per slice usually matches ANOM.3, so provider ranking often stays the same

---

### ANOM.3.a.1 (robust + size-aware) ✅ Recommended primary product

This is ANOM.3.a plus slice-size awareness.
- Adds:
  - slice_n = size(HCPCS_Cd, Year)
  - requires slice_n >= 50 inside the anomalous-row definition
- Aggregates and ranks like ANOM.3.a

✅ Best defensibility: avoids “winner of a tiny peer group” driving provider rank  
✅ Best stability: fewer false positives caused by thin slices  
⚠️ Tradeoff: drops rare-code slices intentionally (feature, not bug)

---

### ANOM.3.b (magnitude-aware provider anomalies; severity-first)

This variant is intentionally different: it emphasizes severity, not just percentile rank.
- Keeps peer extremeness:
  - log_oe_pct_in_slice >= 0.99
- Adds global severity cutoff:
  - log_oe >= quantile(BASE, 0.995) where BASE = over-expected rows
- Provider ranking adds severity bump:
  - + 0.10 * p95_log_oe_mag (severity summary)

✅ Good: produces a “severity-first provider list” for extreme shocks  
⚠️ Weakness: without slice-size filtering it can still admit tiny-slice outliers that look dramatic but are less defensible

---

### ANOM.3.b.1 (magnitude-aware + size-aware) ✅ Recommended alternative product

This is ANOM.3.b plus slice-size awareness (same slice_n >= 50 rule).

✅ Best alternative view: “who has the most severe tail events, in credible peer contexts?”  
✅ More stakeholder-friendly for “big shock” review  
⚠️ Can yield fewer anomalies per provider (often 1–3), so it’s more of a targeted worklist than a “repeat offenders” list

---

## 📊 Comparison table (granular)

| Variant | Core over-expected signal | Peer extremeness definition | Score type | Directional filter | Confidence rule | Denominator hygiene | Slice-size awareness | Severity-aware | Strengths | Risks / failure modes | Best use |
|---|---|---|---|---|---|---|---|---|---|---|---|
| ANOM.3 | oe_ratio | oe_pct_in_slice >= 0.99 | Count + breadth: `n_anom_rows` + 0.25 * `n_unique_codes` + 0.25 * `n_unique_years` | oe_ratio>1 & residual>0 | is_high_conf OR legacy fast-track | None | No | No | Simple baseline; easy to explain | Tiny slices can dominate; OE can be numerically distorted | Baseline sanity check |
| ANOM.3.a | log_oe | log_oe_pct_in_slice >= 0.99 | Count + breadth: `n_anom_rows` + 0.25 * `n_unique_codes` + 0.25 * `n_unique_years` | log_oe>0 & residual>0 | is_high_conf OR legacy fast-track | None | No | No | More numerically stable OE signal | Often yields same set as ANOM.3 (monotonic transform) | “Robust baseline” before size filter |
| ANOM.3.a.1 ✅ | log_oe | log_oe_pct_in_slice >= 0.99, but only for sufficiently large [HCPCS_Cd, Year] slices | Count + breadth: `n_anom_rows` + 0.25 * `n_unique_codes` + 0.25 * `n_unique_years` | log_oe>0 & residual>0 | is_high_conf OR legacy fast-track | None | ✅ slice_n>=50 | No | Most defensible; avoids tiny-slice winners; stable | Drops rare small-slice anomalies (intentional) | Primary provider list (“repeat offenders”) |
| ANOM.3.b | log_oe + severity cutoff | log_oe_pct_in_slice>=0.99 + log_oe>=q(.995) | Count + breadth + severity bump: `n_anom_rows` + 0.25 * `n_unique_codes` + 0.25 * `n_unique_years` + 0.10 * `p95_log_oe_mag` | log_oe>0 & residual>0 | is_high_conf OR legacy fast-track | Optional expected_cost>=MIN_EXPECTED | No | ✅ Yes | “Shock list” view; severity-first | Tiny slices can still create dramatic but weakly supported shocks | Alternative severity-first review |
| ANOM.3.b.1 ✅ | log_oe + severity cutoff | log_oe_pct_in_slice>=0.99 + log_oe>=q(.995), but only for sufficiently large [HCPCS_Cd, Year] slices | Count + breadth + severity bump: `n_anom_rows` + 0.25 * `n_unique_codes` + 0.25 * `n_unique_years` + 0.10 * `p95_log_oe_mag` | log_oe>0 & residual>0 | is_high_conf OR legacy fast-track | Optional | ✅ slice_n>=50 | ✅ Yes | Severity-first but defensible; good worklist | May produce few rows per provider (often 1–3) | Alternative provider product (“big shocks”) |

---

## 🧩 “Two-product” framing (what we now have)

### ✅ Product 1: ANOM.3.a.1 (Primary, robust + size-aware)

- Best for ongoing provider surveillance and “repeat offenders”
- Most defensible to auditors and stakeholders
- Least likely to waste reviewer time on thin-slice artifacts

### ✅ Product 2: ANOM.3.b.1 (Alternative, magnitude-first + size-aware)

- Best for “who has the most severe tail events?”
- Great for targeted deep-dives and escalation review
- Complements the primary list by focusing on severity rather than frequency

---

# Visualizations 

# VIZ.A) "What is an anomaly?" foundation chart (highlight rows that are in primary row list (ANOM.2.a.1))

In [ ]:
# ============================================================
# VIZ.A) "What is an anomaly?" foundation chart
# Scatter: log_oe vs residual
#  - marker/color shows is_high_conf
#  - highlight rows that are in primary row list (ANOM.2.a.1)
#  - reference lines at log_oe=0 and residual=0
# Optional: facet by route (hot_start vs cold_start)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Inputs (expected to exist)
# -----------------------------
# eval_anom: full anomaly universe (eval_scored_DG_V3 + features from ANOM.1)
# anom_top_rows_robust_sizeaware: primary worklist (TOP_N=200) from ANOM.2.a.1

if "eval_anom" not in globals():
    raise NameError("Missing eval_anom. Run ANOM.1 first to create eval_anom.")

if "anom_top_rows_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_rows_robust_sizeaware. Run ANOM.2.a.1 first.")

df = eval_anom.copy()
PRIMARY_ROWS_DF = anom_top_rows_robust_sizeaware

# -----------------------------
# Preconditions / required cols
# -----------------------------
req_cols = ["row_id", "log_oe", "residual", "is_high_conf"]
missing = [c for c in req_cols if c not in df.columns]
if missing:
    raise KeyError(f"eval_anom is missing required columns for this plot: {missing}")

# Ensure numeric
df["log_oe"] = pd.to_numeric(df["log_oe"], errors="coerce")
df["residual"] = pd.to_numeric(df["residual"], errors="coerce")
df["is_high_conf"] = df["is_high_conf"].astype(bool)

# Identify primary rows
primary_ids = set(pd.to_numeric(PRIMARY_ROWS_DF["row_id"], errors="coerce").dropna().astype(int).tolist())
df["is_primary_row"] = pd.to_numeric(df["row_id"], errors="coerce").fillna(-1).astype(int).isin(primary_ids)

# Keep finite
plot_df = df[np.isfinite(df["log_oe"]) & np.isfinite(df["residual"])].copy()

# Optional: clip extreme residuals for readability (toggle)
CLIP_RESID = False
RESID_PCT = 0.999  # clip at 99.9th percentile of abs residual, if enabled
if CLIP_RESID:
    cap = float(np.nanquantile(np.abs(plot_df["residual"].to_numpy()), RESID_PCT))
    plot_df["residual_plot"] = np.clip(plot_df["residual"], -cap, cap)
else:
    plot_df["residual_plot"] = plot_df["residual"]

# -----------------------------
# Plot helpers
# -----------------------------
def _scatter_panel(ax, d: pd.DataFrame, title: str) -> None:
    bg_low = d[(~d["is_high_conf"]) & (~d["is_primary_row"])]
    bg_high = d[(d["is_high_conf"]) & (~d["is_primary_row"])]
    hi = d[d["is_primary_row"]]

    ax.scatter(bg_low["log_oe"], bg_low["residual_plot"], s=8, alpha=0.15, marker="o", label="not high-conf")
    ax.scatter(bg_high["log_oe"], bg_high["residual_plot"], s=10, alpha=0.20, marker="^", label="high-conf")
    ax.scatter(hi["log_oe"], hi["residual_plot"], s=35, alpha=0.9, marker="o", label="PRIMARY (ANOM.2.a.1)")

    ax.axvline(0.0, linewidth=1)
    ax.axhline(0.0, linewidth=1)

    ax.set_title(title)
    ax.set_xlabel("log_oe (log1p(obs) - log1p(exp))")
    ax.set_ylabel("residual (observed - expected)")
    ax.legend(frameon=False, fontsize=9, loc="best")

# -----------------------------
# Option 1: single chart (no faceting)
# -----------------------------
FACET_BY_ROUTE = True  # set False to get just one plot

if not FACET_BY_ROUTE:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    _scatter_panel(ax, plot_df, "What is an anomaly? log_oe vs residual")
    plt.tight_layout()
    plt.show()

# -----------------------------
# Option 2: facet by route (hot_start vs cold_start)
# -----------------------------
else:
    if "route" not in plot_df.columns:
        raise KeyError("FACET_BY_ROUTE=True but eval_anom is missing 'route' column.")

    plot_df["route"] = plot_df["route"].astype(str)

    routes = ["hot_start", "cold_start"]
    other_routes = [r for r in sorted(plot_df["route"].dropna().unique()) if r not in routes]
    routes = routes + other_routes

    n = len(routes)
    fig = plt.figure(figsize=(12, 5 * max(1, int(np.ceil(n / 2)))))

    ncols = 2
    nrows = int(np.ceil(n / ncols))

    for i, r in enumerate(routes, start=1):
        ax = fig.add_subplot(nrows, ncols, i)
        d = plot_df[plot_df["route"] == r]
        _scatter_panel(ax, d, f"Route = {r} (log_oe vs residual)")

    plt.tight_layout()
    plt.show()

Think of VIZ.A as a “sanity + storytelling” chart. It is not the anomaly detector itself. It is a visual check that the worklist produced by ANOM.2.a.1 is behaving the way we claim.

We are visualizing **the entire scored universe** (`eval_anom`, which is `eval_scored_DG_V3` plus the ANOM.1 features), and then **overlaying** the **primary worklist rows** (ANOM.2.a.1) on top.

### **Step 0. Inputs**

- **Universe**: `eval_anom` (about 1.21M rows). Each row is one provider-service-year (`Rndrng_NPI`, `HCPCS_Cd`, `Year`) observation (our `row_id` entity).
- **Primary worklist**: `anom_top_rows_robust_sizeaware`, which is the *top N* rows selected by ANOM.2.a.1.

### **Step 1. Define what x and y mean**

For every row in the universe:

- **x-axis = `log_oe`**
    - Computed as `log1p(observed_cost) - log1p(expected_cost)`.
    - Interpretation:
        - `log_oe > 0` means observed is higher than expected in a multiplicative sense (roughly “over-expected”).
        - `log_oe < 0` means under-expected.
- **y-axis = `residual`**
    - Computed as `observed_cost - expected_cost`.
    - Interpretation:
        - `residual > 0` means observed exceeds expected in absolute dollar terms.
        - `residual < 0` means observed is below expected.

So the plane is split into four quadrants:

- **Top-right (`log_oe > 0`, `residual > 0`)**: over-expected in both multiplicative and absolute terms. This is the “over-cost” direction we care about.
- **Top-left (`log_oe < 0`, `residual > 0`)**: can happen when costs are small but expected is even smaller or if log transform vs absolute is pulling differently (rare, but possible).
- **Bottom-right (`log_oe > 0`, `residual < 0`)**: would suggest log signal says “over” but absolute says “under” (usually indicates near-equality plus numerical effects or inconsistencies, which we fixed).
- **Bottom-left (`log_oe < 0`, `residual < 0`)**: under-expected.

The vertical and horizontal lines at 0 are there to show those quadrants clearly.

### **Step 2. Encode “confidence” visually**

We split (by `is_high_conf`) the universe into:

- not high-conf (background, circles)
- high-conf (background, triangles)

This is only telling the viewer: “these points have enough support (services, benes, tier) to trust more.”

It is not selecting anomalies. It is labeling rows by reliability.

### **Step 3. Overlay the primary anomaly worklist**

We mark rows as `is_primary_row` if `row_id` is in the primary table.

Then we plot them with a bigger marker on top.

This is the key point of the visualization:

- We want to show that **the primary list lives where our definition says it should live**, mainly in the **top-right quadrant**.
- We also want to show that we are not just picking random noise scattered everywhere.

### **Step 4. Optional facet by route**

Faceting by route is a diagnostic:

- **`hot_start`** vs **`cold_start`** behave differently.
- If `cold_start` has heavier tails or more scatter, that is expected and useful to highlight.

### **Step 5. What the figure is demonstrating (the “message”)**

This plot demonstrates three things:

1. **Directionality is correct**: our selected rows are in over-expected territory (top-right).
2. **We are not flagging only “tiny absolute errors”**: many highlighted rows should show meaningful residuals, but not necessarily the most extreme ones (because our method is slice-relative).
3. **Differences by route**: cold_start often has more extreme tails. That’s usually where model anchoring and stability issues show up.

So the plot is basically a “proof-of-behavior” chart for ANOM.2.a.1.

# VIZ.A.1 cell (symlog residual + optional downsample + optional zoomed inset)

In [ ]:
# ============================================================
# VIZ.A.1) "What is an anomaly?" foundation chart (improved)
# Adds:
#   - symlog scaling for residual (handles heavy tails + dense core)
#   - optional background downsample (keeps plot fast/clear)
#   - optional inset zoom near origin (shows dense core structure)
# Facet by route optional (kept)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# -----------------------------
# Inputs (expected to exist)
# -----------------------------
if "eval_anom" not in globals():
    raise NameError("Missing eval_anom. Run ANOM.1 first to create eval_anom.")

if "anom_top_rows_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_rows_robust_sizeaware. Run ANOM.2.a.1 first.")

df = eval_anom.copy()
PRIMARY_ROWS_DF = anom_top_rows_robust_sizeaware

# -----------------------------
# Preconditions / required cols
# -----------------------------
req_cols = ["row_id", "log_oe", "residual", "is_high_conf"]
missing = [c for c in req_cols if c not in df.columns]
if missing:
    raise KeyError(f"eval_anom is missing required columns for this plot: {missing}")

# Ensure numeric
df["log_oe"] = pd.to_numeric(df["log_oe"], errors="coerce")
df["residual"] = pd.to_numeric(df["residual"], errors="coerce")
df["is_high_conf"] = df["is_high_conf"].astype(bool)

# Identify primary rows
primary_ids = set(pd.to_numeric(PRIMARY_ROWS_DF["row_id"], errors="coerce").dropna().astype(int).tolist())
df["is_primary_row"] = pd.to_numeric(df["row_id"], errors="coerce").fillna(-1).astype(int).isin(primary_ids)

# Keep finite
plot_df = df[np.isfinite(df["log_oe"]) & np.isfinite(df["residual"])].copy()

# Optional: clip extreme residuals for readability (toggle)
CLIP_RESID = False
RESID_PCT = 0.999
if CLIP_RESID:
    cap = float(np.nanquantile(np.abs(plot_df["residual"].to_numpy()), RESID_PCT))
    plot_df["residual_plot"] = np.clip(plot_df["residual"], -cap, cap)
else:
    plot_df["residual_plot"] = plot_df["residual"]

# -----------------------------
# Controls
# -----------------------------
USE_SYMLOG_Y = True
SYMLOG_LINTHRESH = 25.0

DOWNSAMPLE_BG = True
BG_MAX = 150_000
RNG_SEED = 7

ADD_INSET = True
INSET_XLIM = (-1.0, 1.0)
INSET_YLIM = (-200.0, 200.0)

def _downsample_keep_primary(d: pd.DataFrame) -> pd.DataFrame:
    if not DOWNSAMPLE_BG:
        return d
    hi = d[d["is_primary_row"]]
    bg = d[~d["is_primary_row"]]
    if len(bg) <= BG_MAX:
        return d
    bg_s = bg.sample(n=BG_MAX, random_state=RNG_SEED)
    return pd.concat([bg_s, hi], axis=0)

INSET_BORDERPAD_HOT = 0.1   # smaller => inset sits closer to the left edge
INSET_BORDERPAD_COLD = 0.1  # keep current look for cold_start

# --- MINIMAL DIFF: expand hot_start x-axis left bound to -6 (match cold_start feel) ---

# (1) Add these controls under our existing inset controls (near INSET_XLIM/INSET_YLIM):
SET_ROUTE_XLIMS = True
HOT_START_XLIM = (-6.0, None)   # None => keep auto right bound
COLD_START_XLIM = (None, None)  # leave cold_start as-is (auto)

# -----------------------------
# Plot helpers
# -----------------------------
def _scatter_panel(ax, d: pd.DataFrame, title: str) -> None:
    d = _downsample_keep_primary(d)

    bg_low = d[(~d["is_high_conf"]) & (~d["is_primary_row"])]
    bg_high = d[(d["is_high_conf"]) & (~d["is_primary_row"])]
    hi = d[d["is_primary_row"]]

    ax.scatter(bg_low["log_oe"], bg_low["residual_plot"], s=8, alpha=0.15, marker="o", label="not high-conf")
    ax.scatter(bg_high["log_oe"], bg_high["residual_plot"], s=10, alpha=0.20, marker="^", label="high-conf")
    ax.scatter(hi["log_oe"], hi["residual_plot"], s=35, alpha=0.9, marker="o", label="PRIMARY (ANOM.2.a.1)")

    ax.axvline(0.0, linewidth=1)
    ax.axhline(0.0, linewidth=1)

    ax.set_title(title)
    ax.set_xlabel("log_oe (log1p(obs) - log1p(exp))")
    ax.set_ylabel("residual (observed - expected)")

    if USE_SYMLOG_Y:
        ax.set_yscale("symlog", linthresh=SYMLOG_LINTHRESH)

    ax.legend(frameon=False, fontsize=9, loc="best")

    # (2) NEW: set x-limits per route, so hot_start gets more left room
    if SET_ROUTE_XLIMS:
        route_label = str(d["route"].iloc[0]) if "route" in d.columns and len(d) else ""
        if route_label == "hot_start":
            cur = ax.get_xlim()
            left = HOT_START_XLIM[0] if HOT_START_XLIM[0] is not None else cur[0]
            right = HOT_START_XLIM[1] if HOT_START_XLIM[1] is not None else cur[1]
            ax.set_xlim(left, right)
        elif route_label == "cold_start":
            # keep as-is or force it later
            if COLD_START_XLIM != (None, None):
                cur = ax.get_xlim()
                left = COLD_START_XLIM[0] if COLD_START_XLIM[0] is not None else cur[0]
                right = COLD_START_XLIM[1] if COLD_START_XLIM[1] is not None else cur[1]
                ax.set_xlim(left, right)

    if ADD_INSET:
        inset_loc = "lower left"

        route_label = str(d["route"].iloc[0]) if "route" in d.columns and len(d) else ""
        inset_borderpad = INSET_BORDERPAD_HOT if route_label == "hot_start" else INSET_BORDERPAD_COLD

        axins = inset_axes(ax, width="17%", height="65%", loc=inset_loc, borderpad=inset_borderpad)
        axins.scatter(bg_low["log_oe"], bg_low["residual_plot"], s=6, alpha=0.12, marker="o")
        axins.scatter(bg_high["log_oe"], bg_high["residual_plot"], s=7, alpha=0.18, marker="^")
        axins.scatter(hi["log_oe"], hi["residual_plot"], s=18, alpha=0.9, marker="o")

        axins.axvline(0.0, linewidth=1)
        axins.axhline(0.0, linewidth=1)
        axins.set_xlim(*INSET_XLIM)
        axins.set_ylim(*INSET_YLIM)
        axins.set_xticks([])
        axins.set_yticks([])

        mark_inset(ax, axins, loc1=3, loc2=1, fc="none", ec="0.5", linewidth=1)

# -----------------------------
# Option 1: single chart (no faceting)
# -----------------------------
FACET_BY_ROUTE = True

if not FACET_BY_ROUTE:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    _scatter_panel(ax, plot_df, "What is an anomaly? log_oe vs residual")
    # remove tight_layout warning when using inset axes
    if not ADD_INSET:
        plt.tight_layout()
    plt.show()

# -----------------------------
# Option 2: facet by route (hot_start vs cold_start)
# -----------------------------
else:
    if "route" not in plot_df.columns:
        raise KeyError("FACET_BY_ROUTE=True but eval_anom is missing 'route' column.")

    plot_df["route"] = plot_df["route"].astype(str)

    routes = ["hot_start", "cold_start"]
    other_routes = [r for r in sorted(plot_df["route"].dropna().unique()) if r not in routes]
    routes = routes + other_routes

    n = len(routes)
    fig = plt.figure(figsize=(12, 6 * max(1, int(np.ceil(n / 2)))))

    ncols = 2
    nrows = int(np.ceil(n / ncols))

    for i, r in enumerate(routes, start=1):
        ax = fig.add_subplot(nrows, ncols, i)
        d = plot_df[plot_df["route"] == r]
        _scatter_panel(ax, d, f"Route = {r} (log_oe vs residual)")

    # remove tight_layout warning when using inset axes
    if not ADD_INSET:
        plt.tight_layout()
    plt.show()

# VIZ.A.2) Small-multiples: within-slice extremeness (ANOM.2.a.1)

In [ ]:
# ============================================================
# VIZ.A.2) Small-multiples: within-slice extremeness (ANOM.2.a.1)
# Expanded: show 8 / 16 / 24 slices + keep helpful print lines
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Inputs (expected to exist)
# -----------------------------
if "eval_anom" not in globals():
    raise NameError("Missing eval_anom. Run ANOM.1 first to create eval_anom.")

if "anom_top_rows_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_rows_robust_sizeaware. Run ANOM.2.a.1 first.")

df = eval_anom.copy()
worklist = anom_top_rows_robust_sizeaware.copy()

# -----------------------------
# Controls
# -----------------------------
N_PANELS = 16          # <-- set to 8, 16, or 24
N_COLS = 4             # 4 works well for 16, 6 works well for 24
ORDER_BY = "n_in_worklist"   # "n_in_worklist" (best), or "n_slice_extreme", or "n_all"

# Optional: keep panels readable
DOWNSAMPLE_BG = True
BG_MAX = 25_000
RNG_SEED = 7

# Slice definition
slice_cols = ["HCPCS_Cd", "Year"]

# -----------------------------
# Preconditions
# -----------------------------
need = ["row_id", "HCPCS_Cd", "Year", "log_oe", "residual"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"eval_anom missing required cols for VIZ.A.2: {missing}")

# Ensure numeric where needed
df["log_oe"] = pd.to_numeric(df["log_oe"], errors="coerce")
df["residual"] = pd.to_numeric(df["residual"], errors="coerce")

# Worklist membership
work_ids = set(pd.to_numeric(worklist["row_id"], errors="coerce").dropna().astype(int).tolist())
df["in_worklist"] = pd.to_numeric(df["row_id"], errors="coerce").fillna(-1).astype(int).isin(work_ids)

# "Slice extreme" membership (within-slice top 1% by log_oe)
# If log_oe_pct_in_slice isn't in eval_anom, compute it
if "log_oe_pct_in_slice" not in df.columns:
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )
df["is_slice_extreme"] = df["log_oe_pct_in_slice"] >= 0.99

# Keep finite for plotting
plot_df = df[np.isfinite(df["log_oe"]) & np.isfinite(df["residual"])].copy()

# -----------------------------
# Build slice list to plot
# -----------------------------
slice_stats = (
    plot_df.groupby(slice_cols, dropna=False)
      .agg(
          n_all=("row_id", "size"),
          n_slice_extreme=("is_slice_extreme", "sum"),
          n_in_worklist=("in_worklist", "sum"),
      )
      .reset_index()
)

if ORDER_BY not in slice_stats.columns:
    raise ValueError(f"ORDER_BY must be one of {list(slice_stats.columns)}")

slice_stats = slice_stats.sort_values(
    [ORDER_BY, "n_in_worklist", "n_slice_extreme", "n_all"],
    ascending=[False, False, False, False]
)

slice_stats = slice_stats.head(N_PANELS).copy()

# -----------------------------
# Helper: downsample background, keep all extremes + all worklist
# -----------------------------
def _subsample_slice(d: pd.DataFrame) -> pd.DataFrame:
    if not DOWNSAMPLE_BG:
        return d
    keep = d[d["is_slice_extreme"] | d["in_worklist"]]
    bg = d[~(d["is_slice_extreme"] | d["in_worklist"])]
    if len(bg) <= BG_MAX:
        return d
    bg_s = bg.sample(n=BG_MAX, random_state=RNG_SEED)
    return pd.concat([bg_s, keep], axis=0)

# -----------------------------
# Plot
# -----------------------------
n = len(slice_stats)
ncols = int(N_COLS)
nrows = int(np.ceil(n / ncols))

fig = plt.figure(figsize=(4.2 * ncols, 3.6 * nrows))

for i, (_, srow) in enumerate(slice_stats.iterrows(), start=1):
    hcpcs = srow["HCPCS_Cd"]
    year = int(srow["Year"])
    n_all = int(srow["n_all"])
    n_ext = int(srow["n_slice_extreme"])
    n_wl = int(srow["n_in_worklist"])

    print(f"[{i:02d}] HCPCS={hcpcs} | Year={year} | n_all={n_all:,} | n_slice_extreme={n_ext:,} | n_in_worklist={n_wl:,}")

    ax = fig.add_subplot(nrows, ncols, i)
    d = plot_df[(plot_df["HCPCS_Cd"] == hcpcs) & (plot_df["Year"] == year)].copy()
    d = _subsample_slice(d)

    # Blue: all rows in slice (background)
    ax.scatter(d["log_oe"], d["residual"], s=10, alpha=0.20, marker="o")

    # Orange: within-slice extremes (top 1% log_oe)
    d_orange = d[d["is_slice_extreme"] & (~d["in_worklist"])]
    ax.scatter(d_orange["log_oe"], d_orange["residual"], s=18, alpha=0.75, marker="o")

    # Green: worklist rows (global TOP_N, size-aware)
    d_green = d[d["in_worklist"]]
    ax.scatter(d_green["log_oe"], d_green["residual"], s=40, alpha=0.95, marker="o")

    ax.axvline(0.0, linewidth=1)
    ax.axhline(0.0, linewidth=1)
    ax.set_title(f"{hcpcs} | {year}\nall={n_all:,}  slice_top1%={n_ext:,}  in_worklist={n_wl:,}")
    ax.set_xlabel("log_oe")
    ax.set_ylabel("residual")

plt.tight_layout()
plt.show()

### **What VIZ.A.2 is proving, in plain terms**

For each panel (one `HCPCS_Cd`, `Year` slice), we are showing three nested layers of evidence:

1.	Blue dots = the entire peer slice

- Every blue point is one row-level observation at the same grain as our universe (one `Rndrng_NPI` × `HCPCS_Cd` × `Place_Of_Srvc` × `Year` row).
- All points share the same HCPCS code and year, so they are comparable peers by design.

2.	Orange dots = “slice-extreme” by definition

- Orange points are the rows that land in the top 1% of `log_oe` within that slice (`log_oe_pct_in_slice` ≥ 0.99).
- This is the literal operational definition of “extreme within peers” for that slice.
- That is why orange points are near the upper-right end of the slice’s diagonal trend.

3.	Green dots = the rows that actually made the global worklist

- Green points are the subset of rows that are in our global `TOP_N` worklist (`anom_top_rows_robust_sizeaware`, `TOP_N`=200), which itself enforces:
    - directionality (`log_oe` > 0 and `residual` > 0)
    - denominator hygiene (if enabled, `expected_cost` ≥ `MIN_EXPECTED`)
    - slice validity (`slice_n` ≥ 50)
    - ranking by the ANOM.2.a.1 `score`: `0.6*log_oe_pct_in_slice + 0.4*resid_pct_in_slice + 0.2*is_high_conf`

So the panel is not just saying “this point is high.” It’s showing:
    - Where the slice’s tail actually is (orange),
    - and which tail point(s) were strong enough, supported enough, and clean enough to survive the global competition across all slices (green).

Why our print lines strengthen the message

Those `n_all`, `n_slice_extreme`, `n_in_worklist` lines are basically an audit trail:
- `n_all tells us how big the peer set is.
- `n_slice_extreme` should be about ~1% of `n_all` (our numbers match that pattern very cleanly).
- `n_in_worklist`=1 in every panel tells us that in these specific slices, one row from that slice made it into the global `TOP_N`. That’s totally expected when we are plotting slices selected because they contributed worklist rows.

A crisp caption we can use (high impact, accurate)

We can say something like:

“Each panel isolates one HCPCS-year peer group. Blue shows all peer rows. Orange marks the top 1% tail by `log_oe` within that peer group. Green marks the rows that actually entered the global ANOM.2.a.1 worklist after directionality, slice-size validity, and confidence-weighted scoring. This shows our anomaly definition is slice-relative, statistically meaningful (n≥50), and consistent across high-volume codes.”

# VIZ.B) Provider-level “what is an anomaly?” foundation chart (ANOM.3.a.1)

In [ ]:
# ============================================================
# VIZ.B) Provider-level "What is an anomaly?" foundation chart (Option 1)
# Scatter: log_oe vs residual
#  - marker/color shows is_high_conf
#  - highlight rows that are (EXTREME EVENTS) AND belong to top providers (ANOM.3.a.1)
#  - reference lines at log_oe=0 and residual=0
# Optional: facet by route (hot_start vs cold_start)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Inputs (expected to exist)
# -----------------------------
# eval_anom: full anomaly universe (eval_scored_DG_V3 + features from ANOM.1)
# anom_top_providers_robust_sizeaware: primary provider worklist (TOP_N=200) from ANOM.3.a.1

if "eval_anom" not in globals():
    raise NameError("Missing eval_anom. Run ANOM.1 first to create eval_anom.")


if "anom_top_providers_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_providers_robust_sizeaware. Run ANOM.3.a.1 first.")

df = eval_anom.copy()
PRIMARY_PROVIDERS_DF = anom_top_providers_robust_sizeaware

# -----------------------------
# Preconditions / required cols
# -----------------------------
req_cols = ["row_id", "log_oe", "residual", "is_high_conf", "Rndrng_NPI"]
missing = [c for c in req_cols if c not in df.columns]
if missing:
    raise KeyError(f"eval_anom is missing required columns for this plot: {missing}")

# Ensure numeric
df["log_oe"] = pd.to_numeric(df["log_oe"], errors="coerce")
df["residual"] = pd.to_numeric(df["residual"], errors="coerce")
df["is_high_conf"] = df["is_high_conf"].astype(bool)

# Identify primary providers (top providers from ANOM.3.a.1)
primary_provider_ids = set(
    pd.to_numeric(PRIMARY_PROVIDERS_DF["Rndrng_NPI"], errors="coerce")
      .dropna().astype(int).tolist()
)
df["is_primary_provider"] = (
    pd.to_numeric(df["Rndrng_NPI"], errors="coerce").fillna(-1).astype(int).isin(primary_provider_ids)
)

# NEW (Option 1): only highlight EXTREME EVENTS that caused providers to rank
if "is_row_anomalous_robust_sizeaware" not in df.columns:
    raise KeyError(
        "Missing is_row_anomalous_robust_sizeaware in eval_anom. "
        "Run ANOM.3.a.1 first (the cell that defines the anomalous-row flag)."
    )

df["is_primary_provider_event"] = df["is_primary_provider"] & df["is_row_anomalous_robust_sizeaware"].astype(bool)

# Keep finite
plot_df = df[np.isfinite(df["log_oe"]) & np.isfinite(df["residual"])].copy()

# Optional: clip extreme residuals for readability (toggle)
CLIP_RESID = False
RESID_PCT = 0.999  # clip at 99.9th percentile of abs residual, if enabled
if CLIP_RESID:
    cap = float(np.nanquantile(np.abs(plot_df["residual"].to_numpy()), RESID_PCT))
    plot_df["residual_plot"] = np.clip(plot_df["residual"], -cap, cap)
else:
    plot_df["residual_plot"] = plot_df["residual"]

# -----------------------------
# Plot helpers
# -----------------------------
def _scatter_panel(ax, d: pd.DataFrame, title: str) -> None:
    bg_low = d[(~d["is_high_conf"]) & (~d["is_primary_provider_event"])]
    bg_high = d[(d["is_high_conf"]) & (~d["is_primary_provider_event"])]
    hi = d[d["is_primary_provider_event"]]

    ax.scatter(bg_low["log_oe"], bg_low["residual_plot"], s=8, alpha=0.15, marker="o", label="not high-conf")
    ax.scatter(bg_high["log_oe"], bg_high["residual_plot"], s=10, alpha=0.20, marker="^", label="high-conf")
    ax.scatter(hi["log_oe"], hi["residual_plot"], s=30, alpha=0.9, marker="o",
               label="EXTREME EVENTS from TOP PROVIDERS (ANOM.3.a.1)")

    ax.axvline(0.0, linewidth=1)
    ax.axhline(0.0, linewidth=1)

    ax.set_title(title)
    ax.set_xlabel("log_oe (log1p(obs) - log1p(exp))")
    ax.set_ylabel("residual (observed - expected)")
    ax.legend(frameon=False, fontsize=9, loc="best")

# -----------------------------
# Option 1: single chart (no faceting)
# -----------------------------
FACET_BY_ROUTE = True  # set False to get just one plot

if not FACET_BY_ROUTE:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    _scatter_panel(ax, plot_df, "Provider-level view: extreme events from top providers (ANOM.3.a.1)")
    plt.tight_layout()
    plt.show()

# -----------------------------
# Option 2: facet by route (hot_start vs cold_start)
# -----------------------------
else:
    if "route" not in plot_df.columns:
        raise KeyError("FACET_BY_ROUTE=True but eval_anom is missing 'route' column.")

    plot_df["route"] = plot_df["route"].astype(str)

    routes = ["hot_start", "cold_start"]
    other_routes = [r for r in sorted(plot_df["route"].dropna().unique()) if r not in routes]
    routes = routes + other_routes

    n = len(routes)
    fig = plt.figure(figsize=(12, 5 * max(1, int(np.ceil(n / 2)))))

    ncols = 2
    nrows = int(np.ceil(n / ncols))

    for i, r in enumerate(routes, start=1):
        ax = fig.add_subplot(nrows, ncols, i)
        d = plot_df[plot_df["route"] == r]
        _scatter_panel(ax, d, f"Route = {r} (extreme events from top providers highlighted)")

    plt.tight_layout()
    plt.show()

# VIZ.B Visualization Interpretation Guide

VIZ.B (Option 1) is doing something very specific. 

Below is what we are visualizing, step by step, mapped directly to our code and to ANOM.3.a.1.

---

### What VIZ.B is visualizing (Option 1), step by step

#### 0) The universe being plotted
**df = eval_anom.copy()**
`eval_anom` is our full row-level universe (same grain as `eval_scored_DG_V3`: one row per `(Rndrng_NPI, HCPCS_Cd, Place_Of_Srvc, Year)`).
Each dot in the scatter plot is **one row-level provider-code-year observation**, not a provider summary row.

So the plot is fundamentally a **row-level scatter plot**, even though the story we are telling is “provider-level”.

#### 1) The two axes define “over-expected direction” in two different ways
* **x-axis: log_oe**
  * Measures relative over-expected-ness (ratio-style signal in log form).
  * `log_oe > 0` means observed > expected in the multiplicative sense.
  * More to the right means “more over-expected” in relative terms.
* **y-axis: residual**
  * Measures absolute over-expected-ness in dollars.
  * `residual > 0` means observed > expected in absolute dollars.
  * Higher up means bigger absolute dollars above expected.

#### 2) The blue crosshairs show the “directionality gates”
We draw:
`ax.axvline(0.0)` and `ax.axhline(0.0)`

That partitions the space into 4 quadrants:
* **Upper-right (log_oe > 0, residual > 0):** Over-expected in both relative and absolute terms. This is the “direction we care about” for our anomaly worklists.
* **Upper-left (log_oe < 0, residual > 0):** Weird edge case. It can happen when expected and observed are both near 0 and transforms behave differently, but in our canonicalized world it should be rare and generally not our target.
* **Lower-right (log_oe > 0, residual < 0):** Also generally inconsistent for properly aligned scoring signals. Should be rare after our corrected consistency checks.
* **Lower-left (log_oe < 0, residual < 0):** Under-expected direction (not the anomaly direction we’re surfacing here).

So the crosshairs are not decoration. They visually encode the basic anomaly direction definition.

#### 3) The background dots show the full distribution (what “normal” looks like)
We split the non-highlight points into two layers:

**A) Blue circles: “not high-conf”**
`bg_low = d[(~d["is_high_conf"]) & (~d["is_primary_provider_event"])]`
These are all rows that are:
1. not in the green set, and
2. not high confidence.
They show the “full noisy cloud”, and they explain why we needed confidence gates and percentiles.

**B) Orange triangles: “high-conf”**
`bg_high = d[(d["is_high_conf"]) & (~d["is_primary_provider_event"])]`
These are all rows that are:
1. not in the green set, but
2. high confidence.
This layer demonstrates that “high confidence” alone does not define anomaly. We still see orange points all over, including near the origin. That is expected.

This is an important rhetorical point: **we are not flagging everything high-confidence.**

#### 4) The green points are the key message (the “proof layer”)
This is the entire point of Option 1. We define the green membership as:

**Step 4.1: “is this provider in the top provider list?”**
`df["is_primary_provider"] = (Rndrng_NPI in top providers list)`
Top providers list = `anom_top_providers_robust_sizeaware` (TOP_N=200) from ANOM.3.a.1. This list is computed from provider-level aggregation and ranking, but it still refers to providers in the same row-level universe.

**Step 4.2: “is this row an ANOM.3.a.1 extreme event?”**
`df["is_primary_provider_event"] = df["is_primary_provider"] & df["is_row_anomalous_robust_sizeaware"]`
Where `is_row_anomalous_robust_sizeaware` is the exact row definition from ANOM.3.a.1. A row is an “extreme event” if:
* `log_oe > 0`
* `residual > 0`
* (`is_high_conf` OR `high_confidence_anomaly_candidate`)
* `log_oe_pct_in_slice >= 0.99` within `(HCPCS_Cd, Year)`
* `slice_n >= 50` (slice-size validity)

So the green points are not “rows from top providers”. They are the **rows that literally satisfied the extreme-event definition**, restricted to providers that rank in the top provider list.

#### 5) Why the green points may still look “not super dramatic”
Even though green is “extreme”, it is **extreme within slice**, not necessarily extreme globally.

Key subtlety:
* `log_oe_pct_in_slice >= 0.99` means: “top 1% among peers that share (HCPCS_Cd, Year)”.
* Some slices (HCPCS-year combinations) might be tight. Their 99th percentile could be modest in absolute terms.
* Also, our confidence gate can allow some rows with small absolute residuals if they are extreme in the peer context.

So green is guaranteed to be in the “right direction”, but not guaranteed to be globally huge on the y-axis.

#### 6) Why faceting by route matters
We split into **hot_start panel** and **cold_start panel**. This is not just cosmetic. It helps the audience see:
* `cold_start` often has wider tails or different shapes because the model and anchor logic differ.
* our extreme-event definition behaves consistently across routes, but the distribution can differ.

---

### The message VIZ.B (Option 1) is designed to deliver

If we had to say it in one sentence:
> “The green points are the exact row-level extreme events (by our ANOM.3.a.1 definition) that caused the top providers to rank highly.”

**What this establishes visually:**
1. We have a clear operational definition of “extreme event”.
2. The provider ranking is not arbitrary. It is built from those events.
3. Those events live in the expected region (primarily upper-right quadrant), so the definition matches the directional intent.

---

### If we want it to “pop” more
Two easy upgrades that strengthen the signal for an audience:
1. Use the VIZ.B.1 version (symlog + inset) so they can see both dense core and tails.
2. Add a small annotation in the caption: **“Green = (top provider) AND (ANOM.3.a.1 extreme event).”**

If we want, paste our VIZ.B.1 output next and I’ll suggest the most impact-oriented caption and 1–2 small visual tweaks (without changing our code structure).

# VIZ.B.1) Provider-level improved chart (symlog residual + downsample + inset), plus hot_start xlim = -6

In [ ]:
# ============================================================
# VIZ.B.1) Provider-level improved chart (ANOM.3.a.1, Option 1)
# Mirrors VIZ.A.1 style:
#   - symlog scaling for residual (handles heavy tails + dense core)
#   - optional background downsample (keeps plot fast/clear)
#   - optional inset zoom near origin (shows dense core structure)
#   - hot_start xlim left expansion to -6 (match cold_start feel)
#
# Green layer:
#   EXTREME EVENTS from TOP PROVIDERS only
#   = (Rndrng_NPI in anom_top_providers_robust_sizeaware) AND (is_row_anomalous_robust_sizeaware == True)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# -----------------------------
# Inputs (expected to exist)
# -----------------------------
if "eval_anom" not in globals():
    raise NameError("Missing eval_anom. Run ANOM.1 first to create eval_anom.")

if "anom_top_providers_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_providers_robust_sizeaware. Run ANOM.3.a.1 first.")

df = eval_anom.copy()
PRIMARY_PROVIDERS_DF = anom_top_providers_robust_sizeaware

# -----------------------------
# Preconditions / required cols
# -----------------------------
req_cols = ["row_id", "log_oe", "residual", "is_high_conf", "Rndrng_NPI"]
missing = [c for c in req_cols if c not in df.columns]
if missing:
    raise KeyError(f"eval_anom is missing required columns for this plot: {missing}")

# Ensure anomalous-row flag exists (must be persisted into eval_anom)
if "is_row_anomalous_robust_sizeaware" not in df.columns:
    raise KeyError(
        "Missing is_row_anomalous_robust_sizeaware in eval_anom. "
        "Persist it from ANOM.3.a.1 (as we did) before running VIZ.B.1."
    )

# Ensure numeric
df["log_oe"] = pd.to_numeric(df["log_oe"], errors="coerce")
df["residual"] = pd.to_numeric(df["residual"], errors="coerce")
df["is_high_conf"] = df["is_high_conf"].astype(bool)

# Identify top providers (from ANOM.3.a.1 output)
primary_provider_ids = set(
    pd.to_numeric(PRIMARY_PROVIDERS_DF["Rndrng_NPI"], errors="coerce").dropna().astype(int).tolist()
)
df["is_primary_provider"] = (
    pd.to_numeric(df["Rndrng_NPI"], errors="coerce").fillna(-1).astype(int).isin(primary_provider_ids)
)

# Green = (top provider) AND (extreme event by ANOM.3.a.1 definition)
df["is_primary_provider_event"] = df["is_primary_provider"] & df["is_row_anomalous_robust_sizeaware"].astype(bool)

# Keep finite
plot_df = df[np.isfinite(df["log_oe"]) & np.isfinite(df["residual"])].copy()

# Optional: clip extreme residuals for readability (toggle)
CLIP_RESID = False
RESID_PCT = 0.999
if CLIP_RESID:
    cap = float(np.nanquantile(np.abs(plot_df["residual"].to_numpy()), RESID_PCT))
    plot_df["residual_plot"] = np.clip(plot_df["residual"], -cap, cap)
else:
    plot_df["residual_plot"] = plot_df["residual"]

# -----------------------------
# Controls (mirror VIZ.A.1)
# -----------------------------
USE_SYMLOG_Y = True
SYMLOG_LINTHRESH = 25.0

DOWNSAMPLE_BG = True
BG_MAX = 150_000
RNG_SEED = 7

ADD_INSET = True
INSET_XLIM = (-1.0, 1.0)
INSET_YLIM = (-200.0, 200.0)

# Expand hot_start x-axis left bound to -6 (match cold_start feel)
SET_ROUTE_XLIMS = True
HOT_START_XLIM = (-6.0, None)   # None => keep auto right bound
COLD_START_XLIM = (None, None)  # leave cold_start as-is (auto)

INSET_BORDERPAD_HOT = 0.1
INSET_BORDERPAD_COLD = 0.1

def _downsample_keep_events(d: pd.DataFrame) -> pd.DataFrame:
    """Downsample background while keeping ALL green (event) rows."""
    if not DOWNSAMPLE_BG:
        return d
    hi = d[d["is_primary_provider_event"]]
    bg = d[~d["is_primary_provider_event"]]
    if len(bg) <= BG_MAX:
        return d
    bg_s = bg.sample(n=BG_MAX, random_state=RNG_SEED)
    return pd.concat([bg_s, hi], axis=0)

# -----------------------------
# Plot helpers
# -----------------------------
def _scatter_panel(ax, d: pd.DataFrame, title: str) -> None:
    d = _downsample_keep_events(d)

    bg_low = d[(~d["is_high_conf"]) & (~d["is_primary_provider_event"])]
    bg_high = d[(d["is_high_conf"]) & (~d["is_primary_provider_event"])]
    hi = d[d["is_primary_provider_event"]]

    ax.scatter(bg_low["log_oe"], bg_low["residual_plot"], s=8, alpha=0.15, marker="o", label="not high-conf")
    ax.scatter(bg_high["log_oe"], bg_high["residual_plot"], s=10, alpha=0.20, marker="^", label="high-conf")
    ax.scatter(
        hi["log_oe"], hi["residual_plot"],
        s=35, alpha=0.9, marker="o",
        label="EXTREME EVENTS from TOP PROVIDERS (ANOM.3.a.1)"
    )

    ax.axvline(0.0, linewidth=1)
    ax.axhline(0.0, linewidth=1)

    ax.set_title(title)
    ax.set_xlabel("log_oe (log1p(obs) - log1p(exp))")
    ax.set_ylabel("residual (observed - expected)")

    if USE_SYMLOG_Y:
        ax.set_yscale("symlog", linthresh=SYMLOG_LINTHRESH)

    # Set x-limits per route (hot_start gets more left room)
    if SET_ROUTE_XLIMS:
        route_label = str(d["route"].iloc[0]) if "route" in d.columns and len(d) else ""
        if route_label == "hot_start":
            cur = ax.get_xlim()
            left = HOT_START_XLIM[0] if HOT_START_XLIM[0] is not None else cur[0]
            right = HOT_START_XLIM[1] if HOT_START_XLIM[1] is not None else cur[1]
            ax.set_xlim(left, right)
        elif route_label == "cold_start":
            if COLD_START_XLIM != (None, None):
                cur = ax.get_xlim()
                left = COLD_START_XLIM[0] if COLD_START_XLIM[0] is not None else cur[0]
                right = COLD_START_XLIM[1] if COLD_START_XLIM[1] is not None else cur[1]
                ax.set_xlim(left, right)

    ax.legend(frameon=False, fontsize=9, loc="best")

    # Inset zoom (dense core near origin) 
    if ADD_INSET:
        inset_loc = "lower left"
        route_label = str(d["route"].iloc[0]) if "route" in d.columns and len(d) else ""
        inset_borderpad = INSET_BORDERPAD_HOT if route_label == "hot_start" else INSET_BORDERPAD_COLD

        axins = inset_axes(ax, width="17%", height="65%", loc=inset_loc, borderpad=inset_borderpad)
        axins.scatter(bg_low["log_oe"], bg_low["residual_plot"], s=6, alpha=0.12, marker="o")
        axins.scatter(bg_high["log_oe"], bg_high["residual_plot"], s=7, alpha=0.18, marker="^")
        axins.scatter(hi["log_oe"], hi["residual_plot"], s=18, alpha=0.9, marker="o")

        axins.axvline(0.0, linewidth=1)
        axins.axhline(0.0, linewidth=1)
        axins.set_xlim(*INSET_XLIM)
        axins.set_ylim(*INSET_YLIM)
        axins.set_xticks([])
        axins.set_yticks([])

        mark_inset(ax, axins, loc1=3, loc2=1, fc="none", ec="0.5", linewidth=1)

# -----------------------------
# Option 1: single chart (no faceting)
# -----------------------------
FACET_BY_ROUTE = True

if not FACET_BY_ROUTE:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    _scatter_panel(ax, plot_df, "Provider-level view: extreme events from top providers (ANOM.3.a.1)")
    if not ADD_INSET:
        plt.tight_layout()
    plt.show()

# -----------------------------
# Option 2: facet by route (hot_start vs cold_start)
# -----------------------------
else:
    if "route" not in plot_df.columns:
        raise KeyError("FACET_BY_ROUTE=True but eval_anom is missing 'route' column.")

    plot_df["route"] = plot_df["route"].astype(str)

    routes = ["hot_start", "cold_start"]
    other_routes = [r for r in sorted(plot_df["route"].dropna().unique()) if r not in routes]
    routes = routes + other_routes

    n = len(routes)
    fig = plt.figure(figsize=(12, 6 * max(1, int(np.ceil(n / 2)))))

    ncols = 2
    nrows = int(np.ceil(n / ncols))

    for i, r in enumerate(routes, start=1):
        ax = fig.add_subplot(nrows, ncols, i)
        d = plot_df[plot_df["route"] == r]
        _scatter_panel(ax, d, f"Route = {r} (extreme events from top providers only)")

    if not ADD_INSET:
        plt.tight_layout()
    plt.show()

# VIZ.B.2) Small-multiples: provider strictness within-slice (ANOM.3.a.1)

In [ ]:
# ============================================================
# VIZ.B.2) Small-multiples: provider strictness within-slice (ANOM.3.a.1)
# For each (HCPCS_Cd, Year) slice:
#   - Blue: all slice rows
#   - Orange: slice extreme events (is_row_anomalous_robust_sizeaware)
#   - Green: extreme events that belong to TOP providers (ANOM.3.a.1)
# Prints per-slice counts to show strictness and confidence.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Inputs (expected to exist)
# -----------------------------
if "eval_anom" not in globals():
    raise NameError("Missing eval_anom. Run ANOM.1 first to create eval_anom.")

if "anom_top_providers_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_providers_robust_sizeaware. Run ANOM.3.a.1 first.")

df = eval_anom.copy()
topprov = anom_top_providers_robust_sizeaware.copy()

# Require event flag
if "is_row_anomalous_robust_sizeaware" not in df.columns:
    raise NameError(
        "Missing is_row_anomalous_robust_sizeaware in eval_anom. "
        "Persist it from ANOM.3.a.1 (we already did this once)."
    )

# -----------------------------
# Basic prep
# -----------------------------
for c in ["log_oe", "residual"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["is_row_anomalous_robust_sizeaware"] = df["is_row_anomalous_robust_sizeaware"].astype(bool)

topprov_ids = set(pd.to_numeric(topprov["Rndrng_NPI"], errors="coerce").dropna().astype(int).tolist())
df["is_top_provider"] = pd.to_numeric(df["Rndrng_NPI"], errors="coerce").fillna(-1).astype(int).isin(topprov_ids)

# Slice key
slice_cols = ["HCPCS_Cd", "Year"]

# Choose slices to display:
# Option: pick slices that contribute the MOST extreme events among TOP providers (high signal)
df["is_topprov_event"] = df["is_top_provider"] & df["is_row_anomalous_robust_sizeaware"]

slice_rank = (
    df.groupby(slice_cols, dropna=False)
      .agg(
          n_all=("row_id","size"),
          n_extreme_rows=("is_row_anomalous_robust_sizeaware","sum"),
          n_topprov_extreme_rows=("is_topprov_event","sum"),
          n_extreme_providers=("Rndrng_NPI", lambda s: s[df.loc[s.index, "is_row_anomalous_robust_sizeaware"]].nunique()),
          n_topprov_extreme_providers=("Rndrng_NPI", lambda s: s[df.loc[s.index, "is_topprov_event"]].nunique()),
      )
      .reset_index()
      .sort_values(["n_topprov_extreme_rows","n_extreme_rows","n_all"], ascending=False)
)

N_PANELS = 16  # set to 24 if desired
picked = slice_rank.head(N_PANELS).copy()

# -----------------------------
# Plot
# -----------------------------
ncols = 4
nrows = int(np.ceil(N_PANELS / ncols))
fig = plt.figure(figsize=(4.8 * ncols, 4.2 * nrows))

for i, row in enumerate(picked.itertuples(index=False), start=1):
    hcpcs = row.HCPCS_Cd
    year = int(row.Year)

    d = df[(df["HCPCS_Cd"] == hcpcs) & (df["Year"] == year)].copy()
    d = d[np.isfinite(d["log_oe"]) & np.isfinite(d["residual"])]

    # layers
    bg = d[~d["is_row_anomalous_robust_sizeaware"]]
    extreme = d[d["is_row_anomalous_robust_sizeaware"]]
    extreme_topprov = extreme[extreme["is_top_provider"]]

    ax = fig.add_subplot(nrows, ncols, i)

    # blue background
    ax.scatter(bg["log_oe"], bg["residual"], s=10, alpha=0.18, marker="o", label="slice background")

    # orange extreme events
    ax.scatter(extreme["log_oe"], extreme["residual"], s=22, alpha=0.75, marker="o",
               label="slice extreme events")

    # green: extreme events from top providers, with black edge so it pops
    ax.scatter(extreme_topprov["log_oe"], extreme_topprov["residual"], s=55, alpha=0.95, marker="o",
               edgecolors="black", linewidths=0.8,
               label="top providers' extreme events")

    ax.axvline(0.0, linewidth=1)
    ax.axhline(0.0, linewidth=1)

    title = (
        f"{hcpcs} | {year}\n"
        f"all={row.n_all:,}  extreme_rows={row.n_extreme_rows:,}  "
        f"extreme_prov={row.n_extreme_providers:,}\n"
        f"topprov_extreme_prov={row.n_topprov_extreme_providers:,}"
    )
    ax.set_title(title)
    ax.set_xlabel("log_oe")
    ax.set_ylabel("residual")

    # legend only on first panel to reduce clutter
    if i == 1:
        ax.legend(frameon=False, fontsize=9, loc="best")

plt.tight_layout()
plt.show()

# Print summary table for audit trail
display(picked)

What VIZ.B.2 is proving, in plain terms

For each panel (one `HCPCS_Cd`, `Year` slice), we are showing three nested layers of evidence, but now the question is provider-level:

1.	Blue dots = the entire peer slice (row-level universe for that HCPCS-year)

- Every blue point is one row-level observation at our standard grain: `Rndrng_NPI` × `HCPCS_Cd` × `Place_Of_Srvc` × `Year`.
- All points share the same HCPCS code and Year, so they are comparable peers by design.
- The cloud shape is the slice’s natural relationship between `log_oe` and `residual` (often a diagonal because both are driven by observed vs expected).

2.	Orange dots = “slice-extreme events” (row-level) by the ANOM.3.a.1 definition

- Orange points are the rows that satisfy the extreme-event definition inside that slice (our provider-level extreme-event rule): 
    - `log_oe` > 0 and `residual` > 0
    - confidence gate: `is_high_conf` OR `high_confidence_anomaly_candidate`
    - slice-relative tail: `log_oe_pct_in_slice` >= 0.99
    - size validity: `slice_n` >= 50 (our .a.1 requirement)
- So orange is the literal set of events that count as “extreme within peers” under ANOM.3.a.1, not just “large residual” or “large OE” in isolation.

3.	Green dots = “top providers’ extreme events” (the provider-level worklist’s evidence)

- Green points are the subset of orange points where:
    - the row is an extreme event AND
    - `Rndrng_NPI` is in `anom_top_providers_robust_sizeaware` (our top-200 provider list from ANOM.3.a.1).
- This is the key difference vs VIZ.A.2.
    - VIZ.A.2 highlights rows that made the top-row worklist.
    - VIZ.B.2 highlights rows that are extreme events and belong to providers who made the top-provider worklist.
- So green is showing the actual evidence rows that explain why those providers “consistently pop.”

⸻

What the companion table is telling us (and how it maps to the plot)

Our table columns line up directly with the three layers:
- `n_all`: number of blue points in that slice (all row-level observations for that HCPCS-year).
- `n_extreme_rows`: number of orange points. This should be roughly “~1% of slice” after gates, but it can be a bit above/below because we are not using a pure percentile-only filter, we also require directionality and confidence.
- `n_topprov_extreme_rows`: number of green points (orange events whose NPI is in the top-provider list).
- `n_extreme_providers`: number of distinct NPIs represented among the orange points.
- `n_topprov_extreme_providers`: number of distinct NPIs among the green points.

So the table is basically an audit trail of:
How many extreme events exist in the slice, how many providers generate them, and how many of those providers are in the top-provider cohort.

Example readout from our table:
- `36415` | `2022`: `n_extreme_rows`=38, `n_extreme_providers`=38 means “almost every extreme event is from a different provider” (one extreme event per provider), and `n_topprov_extreme_providers`=8 means 8 of those providers are in our top-provider list.
- `J3490` | `2022`: `n_extreme_rows`=12, `n_extreme_providers`=12, `n_topprov_extreme_providers`=3 means “12 providers each contribute an extreme event, and 3 of them are top providers.”

That is exactly the provider-level “strictness” story: a provider can rank because they show up repeatedly across slices, even if each slice only has a few extreme events.

⸻

What VIZ.B.2 is demonstrating (the “pop” message)

This figure is not trying to prove “green is in the upper-right quadrant” (that’s true but trivial).

It is proving something more specific:
- Orange shows where the slice’s extreme tail is by our operational definition.
- Green shows how much of that tail is attributable to the top-provider cohort.
- When green appears in many panels, it visually supports: “These providers rank because they repeatedly generate extreme events across multiple HCPCS-year peer groups.”

⸻

A crisp caption we can use (high impact, accurate)

“Each panel isolates a single HCPCS-year peer group. Blue shows all peer rows. Orange marks the ANOM.3.a.1 extreme events within the slice (directional, confidence-gated, top-1% by log_oe, and slice-size valid). Green marks the subset of those extreme events contributed by providers in the top-200 provider worklist. This shows the provider ranking is driven by repeat extreme-event participation across peer slices, not by arbitrary volume alone.”

# SANITY.VIZ.B.2.ONE) Verify VIZ.B.2 counts for ONE (HCPCS, Year) slice

In [ ]:
# ============================================================
# SANITY.VIZB2.ONE) Verify VIZ.B.2 counts for ONE (HCPCS, Year) slice
# FIX: Pull EXPECTED from the VIZ.B.2 table computed in THIS run.
# NOTE: In this notebook, that table is stored as `slice_rank`.
# ============================================================

import pandas as pd
import numpy as np

HCPCS = "36415"
YEAR = 2022

# ---- preconditions ----
assert "eval_anom" in globals(), "Missing eval_anom."
assert "anom_top_providers_robust_sizeaware" in globals(), "Missing anom_top_providers_robust_sizeaware."
assert "slice_rank" in globals(), "Missing slice_rank (our VIZ.B.2 table dataframe)."
assert isinstance(slice_rank, pd.DataFrame), "slice_rank exists but is not a DataFrame."

need_tbl = {"HCPCS_Cd","Year","n_all","n_extreme_rows","n_topprov_extreme_rows","n_extreme_providers","n_topprov_extreme_providers"}
missing_tbl = sorted(list(need_tbl - set(slice_rank.columns)))
assert not missing_tbl, f"slice_rank is missing required columns: {missing_tbl}"

# ---- fetch EXPECTED from slice_rank (same run) ----
row = slice_rank[
    (slice_rank["HCPCS_Cd"].astype(str) == str(HCPCS))
    & (slice_rank["Year"].astype(int) == int(YEAR))
].copy()

assert len(row) == 1, f"Expected exactly 1 VIZ.B.2 row in slice_rank for HCPCS={HCPCS}, Year={YEAR}. Found {len(row)}."

EXPECTED = row[[
    "n_all",
    "n_extreme_rows",
    "n_topprov_extreme_rows",
    "n_extreme_providers",
    "n_topprov_extreme_providers",
]].iloc[0].to_dict()

# ---- compute counts directly from eval_anom ----
d = eval_anom[
    (eval_anom["HCPCS_Cd"].astype(str) == str(HCPCS))
    & (eval_anom["Year"].astype(int) == int(YEAR))
].copy()

assert "is_row_anomalous_robust_sizeaware" in d.columns, "Missing is_row_anomalous_robust_sizeaware in eval_anom."
d["is_row_anomalous_robust_sizeaware"] = d["is_row_anomalous_robust_sizeaware"].astype(bool)

extreme = d[d["is_row_anomalous_robust_sizeaware"]].copy()

top = anom_top_providers_robust_sizeaware.copy()

def norm_npi(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce").astype("Int64")
    return x.astype(str)

# Provider PROFILE key alignment (NPI, provider_type, state)
top["_npi"] = norm_npi(top["Rndrng_NPI"])
top["_pt"] = top["provider_type"].astype(str)
top["_st"] = top["state"].astype(str)
top_keys = set(zip(top["_npi"], top["_pt"], top["_st"]))

extreme["_npi"] = norm_npi(extreme["Rndrng_NPI"])
extreme["_pt"] = extreme["provider_type"].astype(str)
extreme["_st"] = extreme["state"].astype(str)
extreme["prov_key"] = list(zip(extreme["_npi"], extreme["_pt"], extreme["_st"]))

computed = dict(
    n_all=int(len(d)),
    n_extreme_rows=int(len(extreme)),
    n_topprov_extreme_rows=int(extreme["prov_key"].isin(top_keys).sum()),
    n_extreme_providers=int(extreme["prov_key"].nunique()),
    n_topprov_extreme_providers=int(extreme.loc[extreme["prov_key"].isin(top_keys), "prov_key"].nunique()),
)

print("Using expected row from slice_rank (same run).")
print(f"Slice: HCPCS={HCPCS} | Year={YEAR}")
print("Computed:", computed)
print("Expected:", EXPECTED)

for k, v in EXPECTED.items():
    assert computed[k] == int(v), f"Mismatch for {k}: computed={computed[k]} expected={int(v)}"

print("✅ VIZ.B.2 row matches exactly (same-run validation).")

## Why `J3490` and `2022` pattern look different?

In [ ]:
hcpcs = "J3490"  
year = 2022      

d = eval_anom[(eval_anom["HCPCS_Cd"] == hcpcs) & (eval_anom["Year"] == year)].copy()

cols = ["row_id","Rndrng_NPI","route","observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","is_high_conf","high_confidence_anomaly_candidate",
        "is_row_anomalous_robust_sizeaware","guardrail_name","any_guardrail_changed"]
cols = [c for c in cols if c in d.columns]

print("n_rows:", len(d))
print("expected_cost min/median/p1/p99/max:",
      d["expected_cost"].min(), d["expected_cost"].median(),
      d["expected_cost"].quantile(0.01), d["expected_cost"].quantile(0.99),
      d["expected_cost"].max())
print("log_oe max:", d["log_oe"].max())
print("residual max:", d["residual"].max())

# show the biggest offenders
display(d[cols].sort_values("log_oe", ascending=False).head(15))
display(d[cols].sort_values("residual", ascending=False).head(15))

# how much of the slice is near the origin vs dominated by outliers?
display(d[["log_oe","residual","expected_cost","observed_cost"]].describe(percentiles=[0.5,0.9,0.95,0.99,0.999]))

1) What J3490 | 2022 looks like, numerically

Slice size: n_rows = 1,496 (so it passes slice_n >= 50, no issue there).

Expected and observed have a “two-regime” structure
- `expected_cost`
	- median = 0.7186
	- 99th percentile = 17.37
	- 99.9th percentile = 585.81
	- max = 1359.33
- `observed_cost`
	- median = 0.5769
	- 99th percentile = 161.60
	- 99.9th percentile = 1737.82
	- max = 4354.35

That jump from p99 to p99.9 is enormous. That’s already telling us “most rows live in a small-cost world, but a few rows live in a completely different world.”

`log_oe` and `residual` show the same “cliff”
- `log_oe`
	- median = -0.0809 (most rows slightly under expected)
	- 95th percentile = 0.1806
	- 99th percentile = 2.5341
	- 99.9th percentile = 4.7512
	- max = 6.5677
- residual
	- median = -0.1318
	- 95th percentile = 0.3275
	- 99th percentile = 110.96
	- 99.9th percentile = 979.48
	- max = 4320.81

So the distribution is basically:
- a dense cloud near (`log_oe` ~ 0, `residual` ~ 0), with lots of slightly negative residual rows
- plus a handful of rows that are wildly over-expected, driving huge positive residuals and huge `log_oe`

That is exactly why the panel’s autoscaling makes it look different.

2) Why the panel looks different than the others

The panel is dominated by a few “monster” points

Example: the max `log_oe` row:
- `observed_cost` = 1200.02
- `expected_cost` = 0.687
- `residual` ≈ 1199.33
- `oe_ratio` ≈ 1745.7
- `log_oe` ≈ 6.57

This single point forces x to extend far right, and y to extend very high.

And the max residual row:
- `observed_cost` = 4354.35
- `expected_cost` = 33.54
- `residual` ≈ 4320.81
- `log_oe` ≈ 4.84

Again, y has to extend extremely high.

When we include these in a standard linear axis plot, the rest of the slice looks like a thin diagonal ribbon, because the plot has to fit both:
- the dense core, and
- the extreme tail

Most other HCPCS-year slices we plotted did not have tails this extreme (or at least not as extreme relative to their core), so their axes were tighter and looked “normal.”


3. Why `J3490` | `2022` looks different in VIZ.B.2
- `J3490` | `2022` has a strong “two-regime” distribution.
- Most rows cluster near the origin (median `log_oe` ≈ -0.08, median `residual` ≈ -0.13).
- A small tail is extremely over-expected (`log_oe` max ≈ 6.57, `residual` max ≈ 4321).
- Because the panel uses shared linear axes within the panel, those few tail points force very wide x and y limits.
- The wide limits compress the dense core, making the slice look visually different from slices whose tails are less extreme.
- This pattern is consistent with J3490 behaving like a heterogeneous catch-all code, where a small number of rows represent very large-cost events.

 ### A quick “not catastrophic” validation 

In [ ]:
hcpcs="J3490"; year=2022
d = eval_anom[(eval_anom["HCPCS_Cd"]==hcpcs) & (eval_anom["Year"]==year)].copy()

# extreme tail only
tail = d.sort_values("log_oe", ascending=False).head(50)

cols = ["row_id","Rndrng_NPI","route","services","benes","expected_cost_support_tier",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "is_high_conf","high_confidence_anomaly_candidate","any_guardrail_changed","guardrail_name"]
cols = [c for c in cols if c in tail.columns]
display(tail[cols])
print(tail["expected_cost_support_tier"].value_counts(dropna=False) if "expected_cost_support_tier" in tail.columns else "no tier col")

> Comment: This looks like real, systematic signal, not “fragility from tiny denominators” or a plotting artifact.

Why this looks systematic (not fragile)

1) Support is mostly `high` or `medium_high`

In the top tail rows we showed:
- Many are `expected_cost_support_tier` = `high` (hot_start) or `medium_high` (cold_start).
- That means these are not “low-support, shaky” predictions.

2) `services` and `benes` are not tiny

The biggest offenders have meaningful volume, for example:
- `services`=156, `benes`=29 with `log_oe`=6.57
- `services`=72, `benes`=19 with `residual`=4320.8
- `services`=104, `benes`=68 with `log_oe`=4.66
- `services`=109, `benes`=74 with `log_oe`=3.75
Even when some have smaller counts (like `services` in the teens), they are mixed in with many clearly non-trivial ones. So the tail is not driven purely by single-beneficiary one-offs.

3) The tail is “wide” across NPIs and routes

The top 50 includes:
- multiple distinct NPIs (not just one provider)
- both `hot_start` and `cold_start`
That’s a classic signature of a slice-level heavy tail, not an isolated anomaly.

4) `expected_cost` is not near-zero in the worst cases

The extreme points are not “`expected_cost` ~ 0” explosions.
- The worst `log_oe` case has `expected_cost` ≈ 0.687 (small-ish, but not near zero)
- The worst `residual` case has `expected_cost` ≈ 33.54 (not small at all)
So this isn’t the classic “denominator hygiene” failure mode.

5) The flags align with our anomaly logic

Many of these rows have:
- high_confidence_anomaly_candidate = True
- often is_high_conf = True
= and is_row_anomalous_robust_sizeaware = True for the most extreme ones

So our pipeline is behaving consistently: the same rows look extreme by magnitude and also pass confidence gating.

# VIZ.C) Slice-size sanity chart (two-product: ANOM.2.a.1 vs ANOM.2.b.1)

In [ ]:
# ============================================================
# VIZ.C) Slice-size sanity chart (two-product: ANOM.2.a.1 vs ANOM.2.b.1)
# Histogram of slice_n for rows in each worklist
#   - Primary: ANOM.2.a.1  -> anom_top_rows_robust_sizeaware
#   - Alternative: ANOM.2.b.1 -> anom_top_rows_mag_sizeaware
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Inputs (expected to exist)
# -----------------------------
if "eval_anom" not in globals():
    raise NameError("Missing eval_anom. Run ANOM.1 first.")

if "anom_top_rows_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_rows_robust_sizeaware. Run ANOM.2.a.1 first.")

if "anom_top_rows_mag_sizeaware" not in globals():
    raise NameError("Missing anom_top_rows_mag_sizeaware. Run ANOM.2.b.1 first.")

df = eval_anom.copy()
wl_primary = anom_top_rows_robust_sizeaware
wl_alt = anom_top_rows_mag_sizeaware

# -----------------------------
# Ensure slice_n exists in eval_anom (compute if missing)
# -----------------------------
need_cols = ["row_id", "HCPCS_Cd", "Year"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise KeyError(f"eval_anom missing required columns for slice_n: {missing}")

if "slice_n" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")

row_slice_n = df[["row_id", "slice_n"]].copy()
row_slice_n["row_id"] = pd.to_numeric(row_slice_n["row_id"], errors="coerce")
row_slice_n = row_slice_n.dropna(subset=["row_id"])
row_slice_n["row_id"] = row_slice_n["row_id"].astype(int)

def _worklist_slice_n(wl: pd.DataFrame) -> pd.Series:
    if "row_id" not in wl.columns:
        raise KeyError("Worklist missing row_id column.")
    tmp = wl[["row_id"]].copy()
    tmp["row_id"] = pd.to_numeric(tmp["row_id"], errors="coerce")
    tmp = tmp.dropna(subset=["row_id"])
    tmp["row_id"] = tmp["row_id"].astype(int)
    tmp = tmp.merge(row_slice_n, on="row_id", how="left")
    tmp = tmp.dropna(subset=["slice_n"])
    return tmp["slice_n"].astype(int)

s_primary = _worklist_slice_n(wl_primary)
s_alt = _worklist_slice_n(wl_alt)

# -----------------------------
# Plot (log-scaled x)
# -----------------------------
all_vals = pd.concat([s_primary, s_alt], axis=0)
xmin = max(1, int(all_vals.min()))
xmax = int(all_vals.max())

bins = np.unique(np.round(np.logspace(np.log10(xmin), np.log10(xmax), 40)).astype(int))
bins = bins[bins >= 1]
if len(bins) < 10:
    bins = 30

fig = plt.figure(figsize=(12, 7))
ax = fig.add_subplot(111)

ax.hist(s_primary, bins=bins, alpha=0.45, label=f"ANOM.2.a.1 primary (n={len(s_primary):,})")
ax.hist(s_alt, bins=bins, alpha=0.45, label=f"ANOM.2.b.1 alternative (n={len(s_alt):,})")

MIN_SLICE_N = 50
ax.axvline(MIN_SLICE_N, linewidth=2)

ax.set_xscale("log")
ax.set_xlabel("slice_n for rows in worklist (log scale). slice_n = count of rows in (HCPCS_Cd, Year) slice")
ax.set_ylabel("count of worklist rows")
ax.set_title("Slice-size sanity check: both worklists come from sufficiently large peer slices (n ≥ 50)")
ax.legend(frameon=False, fontsize=9, loc="best")

plt.tight_layout()
plt.show()

# -----------------------------
# Quick numeric summary (for narrative)
# -----------------------------
def _summ(name: str, s: pd.Series) -> dict:
    return {
        "worklist": name,
        "n_rows": int(len(s)),
        "min_slice_n": int(s.min()),
        "p10_slice_n": int(np.percentile(s, 10)),
        "median_slice_n": int(np.percentile(s, 50)),
        "p90_slice_n": int(np.percentile(s, 90)),
        "pct_rows_from_slices_lt_50": float((s < 50).mean() * 100),
    }

display(pd.DataFrame([
    _summ("ANOM.2.a.1 primary", s_primary),
    _summ("ANOM.2.b.1 alternative", s_alt),
]))

### **What we’re seeing**

- Vertical blue line at 50
That’s our `slice_n` >= 50 rule. Since both worklists are size-aware, we should see essentially no mass left of 50 (or only tiny numerical edge cases). That part looks right.

- Blue bars (ANOM.2.a.1 primary) spread across a wide range
This is expected because ANOM.2.a.1 is percentile-driven inside each slice (`log_oe_pct_in_slice` + `resid_pct_in_slice` + `is_high_conf`).
Percentile logic does not inherently prefer huge slices. So we get worklist rows coming from many different slice sizes, from ~50 up to the very large slices (992xx etc).

- Orange bars (ANOM.2.b.1 alternative) form a tall “spike” around ~1,000 to ~2,000
Also expected. ANOM.2.b.1 adds a magnitude term (`log_mag`, clipped at `2.0`) that tends to favor slices that produce large absolute shocks and often those come from a smaller number of high-volume, heavy-tail HCPCS-year slices.
If many of oue top 200 “shock” rows come from a handful of slices with similar `slice_n`, they stack into a spike like this.

So the chart is doing its job: it shows our size-aware filter is active, and it visually distinguishes the two products’ “where they source anomalies” behavior.

## Sanity check: histogram counts should sum to TOP_N

### For ANOM.2.a.1

In [ ]:
import numpy as np

prim = anom_top_rows_robust_sizeaware.copy()  # ANOM.2.a.1
bins = np.logspace(np.log10(50), np.log10(prim["slice_n"].max()), 40)

counts, _ = np.histogram(prim["slice_n"].to_numpy(), bins=bins)

print("TOP_N:", len(prim))
print("Sum of histogram bin counts:", int(counts.sum()))
print("Any NaN slice_n in worklist?:", int(prim["slice_n"].isna().sum()))

### For ANOM.2.b.1

In [ ]:
# sanity: histogram counts should sum to TOP_N
import numpy as np

# replace with our actual df name for the alternative worklist
alt = anom_top_rows_mag_sizeaware.copy()  # ANOM.2.b.1

# bins must match what we used in VIZ.C
bins = np.logspace(np.log10(50), np.log10(alt["slice_n"].max()), 40)

counts, _ = np.histogram(alt["slice_n"].to_numpy(), bins=bins)

print("TOP_N:", len(alt))
print("Sum of histogram bin counts:", int(counts.sum()))
print("Any NaN slice_n in worklist?:", int(alt["slice_n"].isna().sum()))

### Confirmation to tie the spike to actual slices

In [ ]:
import pandas as pd

alt = anom_top_rows_mag_sizeaware.copy()  # ANOM.2.b.1 worklist

# How concentrated is the worklist by slice?
by_slice = (
    alt.groupby(["HCPCS_Cd","Year"], dropna=False)
       .size()
       .sort_values(ascending=False)
       .reset_index(name="n_in_worklist")
)

print("Total rows in alt worklist:", len(alt))
print("Top slices contributing rows:")
display(by_slice.head(15))

print("Share of worklist from top 1 / top 3 / top 10 slices:")
top1 = by_slice["n_in_worklist"].head(1).sum() / len(alt)
top3 = by_slice["n_in_worklist"].head(3).sum() / len(alt)
top10 = by_slice["n_in_worklist"].head(10).sum() / len(alt)
print(f"top1={top1:.1%} | top3={top3:.1%} | top10={top10:.1%}")

# Map those top slices to their slice_n values (from our stored slice_n column)
slice_sizes = (
    alt.groupby(["HCPCS_Cd","Year"], dropna=False)["slice_n"]
       .first()
       .reset_index()
)

display(by_slice.head(15).merge(slice_sizes, on=["HCPCS_Cd","Year"], how="left"))

Why the orange histogram has a giant spike (Histogram for ANOM.2.b.1)

1) The orange worklist is dominated by one slice family

Our ANOM.2.b.1 worklist (200 rows total) is overwhelmingly coming from `J3490`:
- `J3490` `2020`: 95 rows (47.5% of the entire worklist)
- `J3490` `2021`: 44 rows
- `J3490` `2023`: 22 rows
- `J3490` `2022`: 20 rows

So just the top 4 slices contribute:
- 95 + 44 + 22 + 20 = 181 rows
- That is 90.5% of the entire `TOP_N=200`

That’s why orange bars are sparse everywhere else. There basically isn’t much else in the magnitude-first list.

2) Why the spike is located around `slice_n` ≈ 1,500

Look at the slice sizes (slice_n) for those J3490 slices:
- `2020`: 1548
- `2021`: 1497
- `2023`: 1509
- `2022`: 1496

So ~181 of our 200 orange points sit in slices with slice_n around ~1,500. When we bin `slice_n` on a log scale, those values fall into the same bin or adjacent bins, producing the big orange spike.

So the spike is not saying “slice_n=1500 causes anomalies.”
It’s saying “the magnitude-first worklist is basically the J3490 tail, and those J3490 slices happen to have slice_n ~ 1500.”

Why the blue histogram does not spike the same way

Our primary ANOM.2.a.1 list is percentile-based and slice-relative, so it tends to pick winners from many different slices rather than being dominated by one HCPCS family. That makes the blue distribution look much more spread out.

The headline we can use in our write-up (accurate + punchy)
- ANOM.2.b.1 is behaving like a “shock detector.” It is dominated by a single HCPCS family (J3490), which produces the biggest absolute log(OE) events.
- ANOM.2.a.1 is behaving like a “broad surveillance detector.” It spreads attention across many HCPCS-year peer slices because it is percentile-first.

# VIZ.D) Provider-level plots

# VIZ.D1.A Repeat offenders, top providers by anomalous-row count (ANOM.3.a.1)

In [ ]:
# ============================================================
# VIZ.D1.A) Repeat offenders (ANOM.3.a.1)
# Bar: Top providers by n_anom_rows_robust
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "anom_top_providers_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_providers_robust_sizeaware. Run ANOM.3.a.1 first.")

df = anom_top_providers_robust_sizeaware.copy()

need = ["Rndrng_NPI","provider_type","state","n_anom_rows_robust","n_rows","anom_rate_pct_robust","n_unique_codes","n_unique_years"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"anom_top_providers_robust_sizeaware missing columns: {missing}")

# Label for plotting (short + readable)
df["prov_label"] = (
    df["Rndrng_NPI"].astype(str).str[-6:] + " | " +
    df["state"].astype(str) + " | " +
    df["provider_type"].astype(str).str.slice(0, 18)
)

TOP = 30
plot_df = df.sort_values("n_anom_rows_robust", ascending=False).head(TOP).copy()
plot_df = plot_df.iloc[::-1]  # reverse for horizontal bar (largest on top)

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111)

ax.barh(plot_df["prov_label"], plot_df["n_anom_rows_robust"].to_numpy())
ax.set_title("ANOM.3.a.1 (Repeat offenders): Top providers by robust extreme-event count")
ax.set_xlabel("n_anom_rows_robust (count of robust extreme events)")
ax.set_ylabel("Provider (last6 NPI | state | type)")

plt.tight_layout()
plt.show()

display(plot_df[["Rndrng_NPI","provider_type","state","n_rows","n_anom_rows_robust","anom_rate_pct_robust","n_unique_codes","n_unique_years"]].iloc[::-1])

### **Repeat offenders bar chart (ANOM.3.a.1)**

What it uses
- `anom_top_providers_robust_sizeaware` (TOP_N=200 providers) from ANOM.3.a.1
- Key columns:
    - `n_anom_rows_robust` = how many robust extreme events this provider generated
    - `n_rows` = provider footprint (how many total rows they have in the universe)
    - `anom_rate_pct_robust` = anomaly rate
    - `n_unique_codes`, `n_unique_years` = breadth/consistency dimensions

What it does
1.	Creates a short readable label per provider: last6(NPI) | state | provider_type
2.	Sorts the top-200 providers by `n_anom_rows_robust`, takes the top 30.
3.	Plots a horizontal bar chart of `n_anom_rows_robust`.

What it’s trying to make pop
- The “repeat offenders” concept in the simplest possible way: which providers have the most robust extreme events (count-based).
- This is not severity. It is frequency of extreme events.

How to read it
- If a provider has a bar of 10, they have 10 rows that satisfied the ANOM.3.a.1 “extreme event” definition (the strict one with `slice_n`>=50 and top 1% `log_oe` in slice, etc.).

Here’s what VIZ.D1.A is showing, step by step, and what it is meant to “pop”.

What D1.A is visualizing

1) What each bar represents
- Each bar = one provider identity at the ANOM.3.a.1 grain, which is:
    - `Rndrng_NPI` + `provider_type` + `state`
- Our y-axis label confirms the plotted label is:
    - last 6 digits of `Rndrng_NPI` | `state` | `provider_type` (truncated)

So we are not “just ranking NPIs”. We are ranking provider records as they appear in our data, stratified by `provider_type` and `state`.

2) What the x-axis measure means
- The x-axis is `n_anom_rows_robust`.
- In ANOM.3.a.1, a row counts toward `n_anom_rows_robust` only if it is an extreme event:

An “extreme event” is a row that satisfies all of these:
- log_oe > 0 (over-expected on log scale)
- residual > 0 (over-expected in absolute dollars)
- confidence gate: is_high_conf OR high_confidence_anomaly_candidate
- within-slice extremeness: log_oe_pct_in_slice >= 0.99 where slice is (HCPCS_Cd, Year)
- slice validity: slice_n >= 50

So `n_anom_rows_robust` = the number of these extreme events attributed to that provider record across all rows they have.

3) What “Top providers by robust extreme-event count” actually means

This plot is purely about repeat frequency.
- A provider with a bar of 10 has 10 separate row-level observations that met the ANOM.3.a.1 extreme-event definition.
- A provider with a bar of 2 has only 2 such events.

This is why the plot is clean and explainable: it is literally “how many extreme events did we generate?”

What it is trying to communicate

The intended message
- These are our “repeat offenders” under the robust + size-aware definition.
- They are not “one-off shocks”. They are providers who repeatedly show up as top-1% within `(HCPCS, Year)` peer slices, with positive residuals, and with confidence support.

Why it “pops”
- We immediately see a steep drop-off: one provider has ~10 events, next ~8, then a cluster around ~6, then ~4, then ~3–2.
- That shape is exactly the “repeat offender” story: a small set of providers generate multiple independent extreme events.

What to say when presenting it (simple caption)

“Each bar is a provider `(NPI, state, provider type)`. The bar height is how many robust, size-validated extreme events that provider generated across all of their rows. This is our repeat-offender view.”

One important nuance (so we do not overclaim)
- This chart does not show severity. A provider could have 10 moderate extremes and rank higher here than a provider with 1 extremely severe outlier.
- That’s fine because this chart is the frequency lens (ANOM.3.a.1).

What each column means in this D1.A context

For each provider record `(Rndrng_NPI, provider_type, state)`:
- `n_rows`: how many total rows this provider has in `eval_anom` (their footprint across all HCPCS-year-POS rows).
- `n_anom_rows_robust`: how many of those rows were flagged as robust + size-aware extreme events (ANOM.3.a.1 definition).
- `anom_rate_pct_robust`: `n_anom_rows_robust / n_rows * 100`. This is their “hit rate”.
- `n_unique_codes`: how many distinct HCPCS codes they appear in.
- `n_unique_years`: how many years they appear in.

So D1.A is not only showing “who has the most extreme events”, it also gives us immediate context for whether those events are happening within a broad practice footprint or a narrow one.

What pops from our top rows

1) The top “repeat offender” by count is also high-rate
- `1851386460` (Heme-Onc, IL): 10 / 159 = 6.29% with 76 codes across 4 years.
- This is both: frequent (10 events) and high-rate (6.3% of their rows).
- Because `n_rows` is only 159, each flagged event moves the rate a lot, but 10 is still a lot.

2) We have two distinct archetypes in the list

A) “High-frequency, smaller footprint”
- Example: `1912911389` (Heme-Onc, NC): 6 / 172 = 3.49%, 92 codes, 3 years.
- Example: `1629282389` (Med Onc, OK): 8 / 272 = 2.94%, 91 codes, 4 years.

These are providers where the flagged events are a meaningful fraction of their activity.

B) “Moderate frequency, massive footprint”
- Example: `1487669933` (Heme-Onc, CA): 7 / 1014 = 0.69%, but 292 codes, 4 years.
- This is a huge footprint provider. Their rate is low, but they still generate multiple extreme events because they touch a lot of code-year space.

This is exactly why having both `n_anom_rows_robust` and `anom_rate_pct_robust` is useful. *Count alone will favor large footprints, rate alone will favor small footprints.*

3) Breadth looks consistently high

Most of these providers have:
- `n_unique_codes` ~ 90–170+
- `n_unique_years` = 4 (almost all)

*That means the “repeat offender” list here is generally not “one-code weirdness”. It is appearing inside a broad coding footprint.*

How to talk about this table in one sentence

“These are the top repeat offenders under the robust size-aware definition. The count column shows how many independent HCPCS-year extreme events each provider generated. The rate, codes, and years columns tell us whether that repetition is concentrated or happening across a broad practice footprint.”

# VIZ.D1.B Repeat offenders, breadth vs frequency (ANOM.3.a.1)

In [ ]:
# ============================================================
# VIZ.D1.B) Repeat offenders (ANOM.3.a.1)
# Scatter: n_unique_codes vs n_anom_rows_robust
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "anom_top_providers_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_providers_robust_sizeaware. Run ANOM.3.a.1 first.")

df = anom_top_providers_robust_sizeaware.copy()

for c in ["n_anom_rows_robust","n_unique_codes","provider_type"]:
    if c not in df.columns:
        raise KeyError(f"Missing {c} in anom_top_providers_robust_sizeaware")

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111)

df["provider_type"] = df["provider_type"].astype(str)
for ptype, g in df.groupby("provider_type", dropna=False):
    ax.scatter(g["n_unique_codes"], g["n_anom_rows_robust"], s=30, alpha=0.6, label=ptype)

ax.set_title("ANOM.3.a.1: Frequency vs breadth (repeat offenders)")
ax.set_xlabel("n_unique_codes (breadth across HCPCS)")
ax.set_ylabel("n_anom_rows_robust (repeat frequency)")
ax.legend(frameon=False, fontsize=9, loc="best")

ax.set_yticks(sorted(df["n_anom_rows_robust"].unique()))

plt.tight_layout()
plt.show()

### **Repeat offenders scatter (breadth vs frequency)**

What it uses
- Same table: `anom_top_providers_robust_sizeaware`

What it does
1.	For each provider in the top-200 repeat-offender list:
	- x-axis = `n_unique_codes` (breadth)
	- y-axis = `n_anom_rows_robust` (frequency)
2.	Colors/legends by `provider_type`.

What it’s trying to make pop
- Whether “repeat offenders” are:
	- narrow + repetitive (high frequency, low breadth), vs
	- broad + systematic (frequency across many codes)

This helps us explain why a provider might be considered concerning:
- “This provider pops across many HCPCS codes” is a different story than
- “This provider pops repeatedly in just one code-family.”

How to read it
- A point high on y but left on x: repeat anomalies but concentrated in few codes.
- A point high on y and far right on x: repeat anomalies across many codes.


D1.B is a “breadth vs frequency” map of our repeat-offender provider worklist (ANOM.3.a.1), and it’s doing one very specific job: separating “providers who pop a lot because they touch a lot of codes” from “providers who pop a lot even with modest breadth.”

Each dot = one provider entity at the grain we used in ANOM.3.a.1: `(Rndrng_NPI, provider_type, state)`

So a provider with the same NPI but different `provider_type` or `state` would appear as a separate dot.

What the axes mean

X-axis: `n_unique_codes`

“How broad is this provider’s footprint?”
- It counts distinct `HCPCS_Cd` values for that provider across all their rows.

Example:
- If a provider billed 120 different HCPCS codes (across years), x = 120.

Y-axis: `n_anom_rows_robust`

“How often does this provider trigger robust extreme events?”
- This is the count of rows where `is_row_anomalous_robust_sizeaware == True` for that provider.

Example:
- If a provider has 6 rows that meet the extreme-event definition, y = 6.

Color: `provider_type`

Just a categorical grouping so we can see whether certain specialties dominate certain regions.

What this plot is trying to make “pop”

We can read this as a simple 2D typology:

1) Upper-left = “repeat offenders with modest breadth”
- High y, lower x.
- Interpretation: “They do not need a huge HCPCS footprint to generate lots of extreme events.”
- *This is usually the most suspicious pattern operationally because frequency is high relative to breadth.*

**In our plot, the topmost point around y=10 is in this zone. That is basically “lots of extreme events despite not being the broadest coder.”**

2) Upper-right = “repeat offenders with huge breadth”
- High y, high x.
- Interpretation: “They generate many extreme events, but they also touch many codes, so some of this may be exposure.”
- *Still worth review, but not as immediately sharp as upper-left.*

We have an obvious point on the far right around x ≈ 290, y ≈ 7. That is “very broad footprint and still many extreme events.”

3) Lower-right = “broad footprint, low frequency”
- Low y, high x.
- Interpretation: “Big practice footprint but only 1–2 extreme events.”
- Often less urgent. Exposure can explain a lot.

4) Lower-left = “small footprint, low frequency”
- Low y, low x.
- Interpretation: “Not really a repeat offender.”

What our specific plot is saying

A few clear takeaways:
1. Most providers cluster at y = 1–3, even among the top 200.
*That’s normal, because once we apply strict extreme-event criteria, most providers only have a small number of qualifying events.*
2. The “repeat offender” story is driven by a small set of points at y ≥ 5.
*Those are the ones we care about for “repeat offenders” messaging.*
3. There are two notable outlier archetypes in our chart:

- A high-frequency, modest-breadth outlier (≈ y=10, x around 70–90).
- A high-frequency, very-broad outlier (≈ y=7, x around 290).

***Those two dots should usually be called out explicitly in narrative because they represent different kinds of operational risk.***

Practical caption we can use

“Each dot is a provider (NPI-type-state). The x-axis shows how many distinct HCPCS codes they billed (breadth). The y-axis shows how many robust extreme events they generated (repeat frequency). Upper-left providers repeat without needing broad exposure. Upper-right providers repeat despite broad exposure.”

In [ ]:
if "anom_top_providers_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_providers_robust_sizeaware. Run ANOM.3.a.1 first.")

df = anom_top_providers_robust_sizeaware.copy()
df = df[df["n_anom_rows_robust"] > 0].copy()

need = [
    "Rndrng_NPI","provider_type","state",
    "n_rows","n_anom_rows_robust","anom_rate_pct_robust",
    "n_unique_codes","n_unique_years",
    "total_services","total_benes","provider_anom_score_robust"
]
need = [c for c in need if c in df.columns]

# Helpful label
df["prov_label"] = (
    df["Rndrng_NPI"].astype(str).str[-6:] + " | " +
    df["state"].astype(str) + " | " +
    df["provider_type"].astype(str).str.slice(0, 24)
)

# ---- 1) Full underlying table (sorted to match how we'd “read” the scatter)
# Primary sort: repeat frequency, then breadth, then volume
sort_cols = [c for c in ["n_anom_rows_robust","n_unique_codes","total_services","n_rows"] if c in df.columns]
df_tbl = df.sort_values(sort_cols, ascending=[False]*len(sort_cols)).copy()

display_cols = ["prov_label"] + need
display(df_tbl[display_cols].head(60))

print("Rows in D1.B scatter (n_anom_rows_robust > 0):", len(df_tbl))

# ---- 2) Identify the key points we want to call out
max_y = df_tbl["n_anom_rows_robust"].max()
max_x = df_tbl["n_unique_codes"].max()

top_freq = df_tbl[df_tbl["n_anom_rows_robust"] == max_y].copy()
top_breadth = df_tbl[df_tbl["n_unique_codes"] == max_x].copy()

print("\nPoint(s) with MAX frequency (highest y):")
display(top_freq[display_cols].head(20))

print("\nPoint(s) with MAX breadth (furthest right x):")
display(top_breadth[display_cols].head(20))

# ---- 3) Optional: “top 10 by frequency” and “top 10 by breadth”
print("\nTop 10 by frequency (y):")
display(df_tbl.sort_values("n_anom_rows_robust", ascending=False)[display_cols].head(10))

print("\nTop 10 by breadth (x):")
display(df_tbl.sort_values("n_unique_codes", ascending=False)[display_cols].head(10))

# VIZ.D1.C Shocks, top providers by severity (ANOM.3.b.1)

In [ ]:
# ============================================================
# VIZ.D1.C) Shocks (ANOM.3.b.1)
# Bar: Top providers by severity (max_log_oe_mag or p95_log_oe_mag)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# If we named it differently, set MAG_TOP_DF = our variable
if "anom_top_providers_mag_sizeaware" in globals():
    MAG_TOP_DF = anom_top_providers_mag_sizeaware.copy()
elif "anom_top_providers_mag" in globals():
    MAG_TOP_DF = anom_top_providers_mag.copy()
else:
    raise NameError("Missing anom_top_providers_mag_sizeaware (or anom_top_providers_mag). Run ANOM.3.b.1 first.")

df = MAG_TOP_DF.copy()

need_any = ["Rndrng_NPI","provider_type","state","n_anom_rows_mag","p95_log_oe_mag","max_log_oe_mag"]
missing = [c for c in need_any if c not in df.columns]
if missing:
    raise KeyError(f"anom_top_providers_mag_sizeaware missing columns: {missing}")

df["prov_label"] = (
    df["Rndrng_NPI"].astype(str).str[-6:] + " | " +
    df["state"].astype(str) + " | " +
    df["provider_type"].astype(str).str.slice(0, 18)
)

# Choose severity axis
SEV_COL = "max_log_oe_mag"   # or "p95_log_oe_mag"

TOP = 30
plot_df = df.sort_values(SEV_COL, ascending=False).head(TOP).copy()
plot_df = plot_df.iloc[::-1]

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111)

ax.barh(plot_df["prov_label"], plot_df[SEV_COL].to_numpy())
ax.set_title(f"ANOM.3.b.1 (Shocks): Top providers by {SEV_COL}")
ax.set_xlabel(SEV_COL + " (peak shock severity)")
ax.set_ylabel("Provider (last6 NPI | state | type)")

plt.tight_layout()
plt.show()

display(plot_df[["Rndrng_NPI","provider_type","state","n_anom_rows_mag","p95_log_oe_mag","max_log_oe_mag"]].iloc[::-1])

# VIZ.D1.D Overlap summary (repeat offenders vs shocks)

In [ ]:
# ============================================================
# VIZ.D1.D) Overlap between the two provider worklists (enhanced)
# ============================================================

import numpy as np
import pandas as pd

if "anom_top_providers_robust_sizeaware" not in globals():
    raise NameError("Missing anom_top_providers_robust_sizeaware. Run ANOM.3.a.1 first.")

if "anom_top_providers_mag_sizeaware" in globals():
    mag_df = anom_top_providers_mag_sizeaware.copy()
elif "anom_top_providers_mag" in globals():
    mag_df = anom_top_providers_mag.copy()
else:
    raise NameError("Missing anom_top_providers_mag_sizeaware (or anom_top_providers_mag). Run ANOM.3.b.1 first.")

rob = anom_top_providers_robust_sizeaware.copy()

KEY = ["Rndrng_NPI","provider_type","state"]
for c in KEY:
    if c not in rob.columns or c not in mag_df.columns:
        raise KeyError(f"Both provider tables must contain {KEY}.")

# --- set sizes (as sets of provider keys) ---
rob_keys = set(map(tuple, rob[KEY].astype(str).to_numpy()))
mag_keys = set(map(tuple, mag_df[KEY].astype(str).to_numpy()))

both = rob_keys & mag_keys
only_rob = rob_keys - mag_keys
only_mag = mag_keys - rob_keys

n_rob = len(rob_keys)
n_mag = len(mag_keys)
n_both = len(both)

print("Repeat-offender list (ANOM.3.a.1) size:", n_rob)
print("Shock list (ANOM.3.b.1) size:", n_mag)
print("Overlap (in both):", n_both)
print("Only repeat offenders:", len(only_rob))
print("Only shocks:", len(only_mag))

# --- NEW: overlap shares ---
print(f"Overlap share of repeat list: {100.0 * n_both / max(n_rob, 1):.1f}%")
print(f"Overlap share of shock list:  {100.0 * n_both / max(n_mag, 1):.1f}%")

# --- build overlap frame with key metrics ---
overlap_df = rob.merge(mag_df, on=KEY, how="inner", suffixes=("_rob","_mag")).copy()

# Coerce numerics safely
for c in ["n_anom_rows_robust","provider_anom_score_robust","p95_log_oe_mag","max_log_oe_mag","n_anom_rows_mag"]:
    if c in overlap_df.columns:
        overlap_df[c] = pd.to_numeric(overlap_df[c], errors="coerce")

# --- NEW: interpretability columns ---
# exp(log_oe) ~ O/E ratio for the worst event, since log_oe = log((1+obs)/(1+exp)) approx log(obs/exp) for larger values
if "max_log_oe_mag" in overlap_df.columns:
    overlap_df["max_oe_ratio_equiv"] = np.exp(overlap_df["max_log_oe_mag"].astype(float))
if "p95_log_oe_mag" in overlap_df.columns:
    overlap_df["p95_oe_ratio_equiv"] = np.exp(overlap_df["p95_log_oe_mag"].astype(float))

# --- Display 1: overlaps sorted by repeat frequency then severity ---
show_cols = [c for c in (
    KEY
    + ["n_anom_rows_robust","provider_anom_score_robust","n_anom_rows_mag",
       "max_log_oe_mag","max_oe_ratio_equiv","p95_log_oe_mag","p95_oe_ratio_equiv"]
) if c in overlap_df.columns]

print("\nOverlap providers (sorted: repeat frequency then shock severity):")
display(
    overlap_df.sort_values(
        ["n_anom_rows_robust","max_log_oe_mag"],
        ascending=[False, False]
    )[show_cols].head(30)
)

# --- Display 2 (optional): high-severity overlap subset ---
SEV_THRESH = 2.0  # log_oe ~ 2 => O/E equiv ~ 7.39
if "max_log_oe_mag" in overlap_df.columns:
    hi = overlap_df[overlap_df["max_log_oe_mag"] >= SEV_THRESH].copy()
    print(f"\nHigh-severity overlaps (max_log_oe_mag >= {SEV_THRESH}) (O/E equiv ~ 7.39): {len(hi)} providers")
    if len(hi):
        display(
            hi.sort_values("max_log_oe_mag", ascending=False)[show_cols].head(30)
        )

#########################################################################################################################################################################################################\ #########################################################################################################################################################################################################\ #########################################################################################################################################################################################################\ #########################################################################################################################################################################################################\ #########################################################################################################################################################################################################\

# COMPREHENSIVE EXECUTIVE SUMMARY 

# Anomaly Surfacing Workflow (Row-level + Provider-level)  
**Project:** Medicare Provider Benchmarking Engine  
**Notebook family:** Anomaly surfacing notebook (downstream of Guardrails notebook)  
**Source of truth:** `artifacts/eval_universe/eval_scored_DG_V3.parquet` (D/G + Guardrails V3 applied, canonicalized scored columns)

---

## 0) Purpose and “two-product” framing

This workflow turns a scored benchmarking universe into **actionable anomaly worklists** at two levels:

1) **Row-level products (what specific provider–service–year observations look suspicious?)**
- **Primary product (recommended):** **ANOM.2.a.1**  
  Robust, slice-relative, size-aware. Designed to be stable and reviewable.
- **Alternative product (contrast / shock list):** **ANOM.2.b.1**  
  Magnitude-first, still slice-aware. Designed to surface the most dramatic “dollar shocks”.

2) **Provider-level products (which providers “consistently pop” as repeat offenders vs shocks?)**
- **Primary product (recommended):** **ANOM.3.a.1**  
  Repeat offenders. Counts how often a provider triggers slice-relative extreme events.
- **Alternative product (contrast / shock list):** **ANOM.3.b.1**  
  Shocks. Ranks providers by severity summaries among anomalous rows only.

This gives us an auditable, defensible “two-product” system:
- **Product A:** repeatable, stable, comparable across slices (primary)
- **Product B:** reveals biggest tail events (alternative)

---

## 1) Data and grain (what a “row” is)

### 1.1 Single source of truth (scored evaluation universe)
- **File:** `artifacts/eval_universe/eval_scored_DG_V3.parquet`
- **What it represents:** one row per **(provider × service × place × year)** with:
  - observed cost (actual)
  - expected cost (predicted, post-guardrail)
  - benchmarking signals (residual, O/E ratio, log signals)
  - routing metadata (hot_start vs cold_start)
  - support tiers and flags

### 1.2 Row grain (confirmed)
The row grain is **unique** at:

- `Rndrng_NPI`
- `HCPCS_Cd`
- `Place_Of_Srvc`
- `Year`

We confirmed:
- `rows == unique(keys)` and `max duplicates per key == 1`  
for both `eval_scored_DG_V3` and `eval_anom`.

So every dot in the row-level visuals and every entry in row-level worklists is one unique:
> **NPI × HCPCS × Place of Service × Year**

---

## 2) Canonical scored columns (critical stability step)

### 2.1 Why canonicalization exists
During debugging, we discovered:
- artifacts from different notebooks (modeling vs guardrails) can contain **different expected_cost** values and inconsistent derived columns unless we enforce a single canonical definition.
- `benchmark_df_rich_v2` and `failure_df_full_v2` are **pre-guardrail baseline artifacts** (D/G scored) created in the modeling notebook.
- `eval_scored_DG_V3` is the **post-guardrail** evaluation artifact (D/G + V3) created in the guardrails notebook.

Therefore:
- `benchmark_df_rich_v2` and `failure_df_full_v2` are not “wrong”. They are baseline snapshots.
- `eval_scored_DG_V3` is the one anomaly surfacing should trust.

### 2.2 Canonicalization cell (before exporting eval_scored_DG_V3)
Right before exporting `eval_scored_DG_V3.parquet`, we recompute:
- `residual = observed_cost - expected_cost`
- `oe_ratio = observed_cost / (expected_cost + EPS)`
- `pct_diff = residual / (expected_cost + EPS)`
- `log_oe = log1p(obs) - log1p(exp)` (log of ratio on the 1+ scale)

This guarantees internal consistency and removes “stale derived columns” issues.

### 2.3 Post-refresh sanity outcome
After refresh and rerun:
- residual consistency checks pass
- oe_ratio consistency checks pass
- log_oe consistency checks pass

This matters because all anomaly definitions rely on:
- directionality (log_oe > 0, residual > 0)
- slice percentiles (log_oe_pct_in_slice)
- ranking stability

---

## 3) ANOM.1: Build the anomaly universe (`eval_anom`)

### 3.1 What ANOM.1 produces
`eval_anom` is the working dataframe for anomaly surfacing. Conceptually:

- `eval_anom = eval_scored_DG_V3`  
  plus additional computed fields used for anomaly detection, including:

**Core scored signals**
- observed_cost
- expected_cost
- residual
- oe_ratio
- log_oe
- pct_diff

**Slice-relative percentiles** (slice defined as `(HCPCS_Cd, Year)`)
- `resid_pct_in_slice`
- `oe_pct_in_slice` (used in baseline variants)
- `log_oe_pct_in_slice` (used in robust variants)

**Confidence metadata**
- `is_high_conf` (tiering based on support, services, benes)
- `high_confidence_anomaly_candidate` (fast-track heuristic)
- `expected_cost_support_tier`
- `route` (hot_start vs cold_start)

**Guardrail metadata**
- `guardrail_name`
- `any_guardrail_changed`

### 3.2 Why the slice is `(HCPCS_Cd, Year)`
This is the comparability design:
- We do not compare different HCPCS codes directly.
- We do not compare across years directly.
- We define peers as “same service, same year”.

So a percentile threshold like “top 1%” is meaningful within a slice.

---

## 4) Row-level anomaly products (ANOM.2 family)

Row-level products produce a TOP_N worklist of suspicious rows.

### Shared building blocks across all row-level variants
1) **Candidate universe**
   - requires non-null core fields (log_oe, residual, observed_cost, expected_cost)
   - optional denominator hygiene (expected_cost >= MIN_EXPECTED)

2) **Directionality (over-expected only)**
   - `log_oe > 0`
   - `residual > 0`

3) **Slice-relative comparison**
   - percentiles computed within `(HCPCS_Cd, Year)` slice

4) **Confidence boost**
   - `is_high_conf` enters scoring as an additive bump

---

### 4.1 ANOM.2 baseline (OE-based percentile)
**Concept:** use raw O/E (`oe_ratio`) and residual percentile to rank.

**Extremeness definition (slice-relative):**
- tail by `oe_pct_in_slice` and `resid_pct_in_slice`
- directionality: oe_ratio > 1 and residual > 0

**Known weakness:**
- OE ratio can blow up when expected_cost is tiny.
- tails can be unstable without denominator hygiene.

---

### 4.2 ANOM.2.a (robust percentile using log_oe)
**Concept:** replace OE tail with log(OE) tail for stability.

**Extremeness definition (slice-relative):**
- `log_oe_pct_in_slice` and `resid_pct_in_slice`
- directionality: log_oe > 0 and residual > 0
- optional expected_cost >= MIN_EXPECTED

**Why it’s better than baseline:**
- log scale compresses the explosive OE tail
- ranking becomes less sensitive to tiny expected denominators

---

### 4.3 ANOM.2.a.1 ✅ Primary row-level product (robust + size-aware)
**Adds slice-size validity** so “top 1%” is meaningful.

**Additional requirement:**
- compute `slice_n = size(HCPCS_Cd, Year)`
- enforce `slice_n >= MIN_SLICE_N` (we used 50)

**Score (exact)**
- `anom_score_robust = 0.6*log_oe_pct_in_slice + 0.4*resid_pct_in_slice + 0.2*is_high_conf`

**Output**
- `anom_top_rows_robust_sizeaware` (TOP_N=200)

**Why this is the primary**
- strongest defensibility
- avoids “winner of a tiny slice”
- stable worklist behavior across reruns

---

### 4.4 ANOM.2.b (magnitude-aware scoring)
**Concept:** still uses slice percentiles, but also rewards *absolute severity* of log_oe.

**Magnitude transform**
- `log_mag = clip(log_oe, 0, LOG_CLIP_MAX) / LOG_CLIP_MAX`  
  (LOG_CLIP_MAX=2.0, oe_ratio ~ 7.39 cap)

**Score (exact)**
- `anom_score_mag = 0.45*log_oe_pct_in_slice + 0.35*resid_pct_in_slice + 0.35*log_mag + 0.15*is_high_conf`

**What it does**
- creates a “shock list” of the biggest tail anomalies
- can concentrate on certain HCPCS families if they dominate severity tails

---

### 4.5 ANOM.2.b.1 ✅ Alternative row-level product (magnitude-aware + size-aware)
Same as ANOM.2.b plus:
- `slice_n >= MIN_SLICE_N` filter

**Output**
- `anom_top_rows_mag_sizeaware` (TOP_N=200)

**Why this is the best alternative**
- still defensible (size-aware)
- but intentionally emphasizes “dramatic” shocks

---

## 5) Provider-level anomaly products (ANOM.3 family)

Provider-level products identify providers who “consistently pop” by aggregating row-level events.

### Shared building blocks across provider-level variants
All provider variants:
1) define a row-level event flag (what counts as an “extreme event”)
2) group by provider identity:
   - `Rndrng_NPI`, `provider_type`, `state`
3) compute:
   - frequency: how many events
   - breadth: how many codes/years involved
   - optional severity summaries (in magnitude variants)
4) rank providers into a TOP_N list

---

### 5.1 ANOM.3 baseline (OE-based)
**Extreme event definition**
- `oe_ratio > 1`
- `residual > 0`
- confidence gate: `is_high_conf OR high_confidence_anomaly_candidate`
- slice tail: `oe_pct_in_slice >= 0.99` within `(HCPCS_Cd, Year)`

**Meaning**
- providers with the most frequent top-1% OE events within their HCPCS-year peer slices

---

### 5.2 ANOM.3.a (robust using log_oe)
Same structure as ANOM.3 but:
- tail is on `log_oe_pct_in_slice` instead of OE percentile
- directionality: `log_oe > 0` and `residual > 0`

**Meaning**
- providers with frequent top-1% log(OE) events within their HCPCS-year peer slices

---

### 5.3 ANOM.3.a.1 ✅ Primary provider-level product (robust + size-aware)
Adds:
- `slice_n >= MIN_SLICE_N` for `(HCPCS_Cd, Year)` slice validity

**Event flag**
- `is_row_anomalous_robust_sizeaware`

**Provider aggregation**
- `n_anom_rows_robust`: count of robust extreme events
- `anom_rate_pct_robust`: event rate among provider’s rows
- breadth: `n_unique_codes`, `n_unique_years`
- volume: services and benes totals

**Provider score**
- `provider_anom_score_robust = n_anom_rows_robust + 0.25*n_unique_codes + 0.25*n_unique_years`

**Output**
- `anom_top_providers_robust_sizeaware` (TOP_N=200)

**Interpretation**
These are “repeat offenders”:
- not just one dramatic event, but repeated slice-relative extremes across codes and years.

---

### 5.4 ANOM.3.b (magnitude-aware provider anomalies)
This variant is intentionally different:
- define anomalous rows using a global severity cutoff on log_oe
- then summarize severity among anomalous rows only

Key change:
- severity metrics like `max_log_oe_mag`, `p95_log_oe_mag` should be computed **only among anomalous rows**, not among all rows for the provider

This is why we changed aggregation to masked functions (groupby apply) so:
- `max_log_oe_mag` and `p95_log_oe_mag` reflect **severity of flagged events**, not baseline provider distribution.

---

### 5.5 ANOM.3.b.1 ✅ Alternative provider-level product (magnitude-aware + size-aware)
Adds:
- `slice_n >= MIN_SLICE_N` for event definition

Ranks providers by:
- frequency of magnitude anomalies
- breadth across codes/years
- plus a small severity bump:
  - `provider_anom_score_mag = n_anom_rows_mag + 0.25*n_unique_codes + 0.25*n_unique_years + 0.10*p95_log_oe_mag`

**Output**
- `anom_top_providers_mag_sizeaware` (TOP_N=200)

**Interpretation**
These are “shock providers”:
- providers associated with the most extreme tail severity events.

---

## 6) Visualization strategy (what each plot proves)

The visualization suite is designed to be:
- understandable to an audience
- auditable (ties directly to definitions)
- aligned with the two-product framing

---

### 6.1 VIZ.A and VIZ.A.1 (row-level directional evidence)
**Goal**
Show the foundational directionality concept:
- over-expected means positive residual and log_oe > 0

**Data**
- background: `eval_anom` (all rows)
- highlight: rows in primary worklist `anom_top_rows_robust_sizeaware`

**What it proves**
- the highlighted anomalies live where they “should” (upper-right by design)
- confidence layering shows we are not surfacing pure noise

**Why VIZ.A.1 exists**
- heavy tails compress the view
- symlog scaling and inset show both:
  - dense core near origin
  - global tails

---

### 6.2 VIZ.A.2 (row-level small multiples, slice-relative proof)
**Goal**
Make “extreme within peers” visually obvious and unambiguous.

For each panel = one `(HCPCS_Cd, Year)` slice:

- **Blue:** all rows in that slice (peer universe)
- **Orange:** slice-extreme tail (top 1% by log_oe percentile)
- **Green:** the row(s) from that slice that made the global TOP_N worklist (primary product)

**What it proves**
- “extreme” is slice-relative, not global
- green points are not arbitrary. They are top-ranked survivors of a strict definition and global competition
- printed audit lines (`n_all`, `n_slice_extreme`, `n_in_worklist`) make the logic checkable

---

### 6.3 VIZ.B (provider-level directional evidence, Option 1)
**Goal**
Avoid ambiguity in provider-level highlighting.

Instead of highlighting “all rows belonging to top providers”, Option 1 highlights:
- only the extreme events that caused providers to rank.

**Green points**
- rows where:
  - `is_row_anomalous_robust_sizeaware == True`
  - AND provider is in top provider list (`anom_top_providers_robust_sizeaware`)

**What it proves**
- provider ranking is driven by repeated genuine extreme events
- green points cluster in the upper-right quadrant by construction, visually proving the event definition

---

### 6.4 VIZ.B.2 (provider strictness within-slice)
**Goal**
Make provider-level strictness tangible and auditable in slice terms.

For each `(HCPCS_Cd, Year)` slice:
- compute:
  - `n_all` (slice size)
  - `n_extreme_rows` (events meeting ANOM.3.a.1 extreme event definition)
  - `n_extreme_providers` (providers contributing those extreme rows)
  - `n_topprov_extreme_rows` (subset of extreme events belonging to top-200 providers)
  - `n_topprov_extreme_providers` (providers among top-200 contributing those)

**What it proves**
- top providers are not chosen arbitrarily. They appear repeatedly inside slice tails
- we can trace top providers back to slice-level events

We validated counts with a one-slice recomputation and exact match.

---

### 6.5 VIZ.C (slice-size sanity, two-product)
**Goal**
Demonstrate defensibility of slice-size filtering (n>=50) and characterize slice distribution in the two worklists.

**Interpretation**
- Blue histogram: distribution of slice sizes across all eligible slices / or across baseline candidate pool (depending on our implementation)
- Orange histogram: slice size distribution of the TOP_N worklist rows (each bar height counts worklist rows)
- Histogram bin count sum = TOP_N (validated)

**Key insight we discovered**
- the magnitude product (ANOM.2.b.1) can concentrate heavily in a small number of slices (example: J3490 across years).  
  That is expected behavior and explains spikes.

---

## 7) Simpler executive-ready provider visuals (VIZ.D1 family)

To make provider story “simple and linear”, we built:

### 7.1 VIZ.D1.A (repeat offenders bar chart)
**Input:** `anom_top_providers_robust_sizeaware`  
**Shows:** top providers by `n_anom_rows_robust`  
**Message:** who triggers the most slice-relative extreme events (frequency)

### 7.2 VIZ.D1.B (repeat offenders breadth vs frequency)
**Input:** `anom_top_providers_robust_sizeaware` filtered to n_anom_rows_robust > 0  
**Shows:** scatter of
- x = `n_unique_codes` (breadth across HCPCS)
- y = `n_anom_rows_robust` (frequency)
**Message:** distinguish:
- narrow repeat offenders (high y, low x)
- broad repeat offenders (high y and high x)

### 7.3 VIZ.D1.C (shocks bar chart)
**Input:** `anom_top_providers_mag_sizeaware`  
**Shows:** top providers by severity metric (`max_log_oe_mag` or `p95_log_oe_mag`)  
**Message:** who has the most extreme single-event shocks

### 7.4 VIZ.D1.D (overlap between repeat offenders and shocks)
**Input:** both top-200 provider lists  
**Shows:**
- overlap count
- overlap share
- overlap table with both frequency and severity, plus oe_ratio equivalents (`exp(log_oe)`)

**Key takeaway**
Repeat offenders and shocks are largely distinct:
- overlap was small (~9.5% in our run)
- that validates the two-product framing at provider level

---

## 8) Final deliverables (what we can export as “executive summary lists”)

### 8.1 Row-level worklists (TOP_N=200)
1) **Primary row list:** `anom_top_rows_robust_sizeaware` (ANOM.2.a.1)  
   Use for routine review and follow-up.
2) **Shock row list:** `anom_top_rows_mag_sizeaware` (ANOM.2.b.1)  
   Use for “largest shocks” and contrast.

### 8.2 Provider-level worklists (TOP_N=200)
1) **Repeat offenders:** `anom_top_providers_robust_sizeaware` (ANOM.3.a.1)  
   Use for compliance review / operational follow-up.
2) **Shock providers:** `anom_top_providers_mag_sizeaware` (ANOM.3.b.1)  
   Use for investigating rare but extreme events.

---

## 9) How to interpret the outputs safely (non-overclaiming)

What these lists **are**:
- statistically and operationally defensible prioritization lists
- slice-relative extremeness within comparable peer groups (HCPCS-year)
- constrained to over-expected direction and confidence gates
- size-aware (n>=50) in primary products

What these lists **are not**:
- proof of fraud
- proof of coding errors
- proof of clinical inappropriateness

They are “worklists”:
> a ranked queue of observations and providers that merit investigation.

---

## 10) Executive takeaways (tight, high-impact)

### Row-level (ANOM.2)
- We define anomalies as **over-expected**, **slice-extreme**, **confidence-gated** observations.
- The primary product is **slice-relative and size-valid**, reducing false positives from tiny peer sets.
- The alternative product finds **biggest magnitude shocks**, and can concentrate in specific HCPCS families, which is expected and transparent.

### Provider-level (ANOM.3)
- We define provider “repeat offenders” as providers with **many slice-extreme events** across codes and years.
- We define provider “shocks” as providers associated with **very high-severity events** (p95/max among anomalous events).
- Overlap between repeat offenders and shocks is small, supporting the need for two products.

### Visual evidence
- Directionality plots show anomalies are in the correct quadrant by definition.
- Small-multiples show slice-relative tails clearly and auditably.
- Slice-size sanity plot proves “top 1%” is meaningful (n>=50).
- Provider visuals separate repeat offenders from shocks cleanly.

---

## 11) Recommended next step (operationalization)
Create a lightweight “case packet” export for each list:
- include slice context (slice_n, slice percentile)
- include confidence and tier metadata
- include guardrail flags
- include top 3 peer comparators within the same slice (for reviewer context)

This turns each list into a review-ready artifact.

# Freezing Worklists (ANOM.2.a.1, ANOM.2.b.1, ANOM.3.a.1, ANOM.3.b.1)

In [ ]:
# ============================================================
# FREEZE.WORKLISTS) Freeze final anomaly worklists to disk
# - Writes: CSV + Parquet for each worklist, timestamped
# - Writes: params.json + params.md capturing exact knobs
# ============================================================

from pathlib import Path
import json
import pandas as pd

# -----------------------------
# 0) REQUIRED: point to the 4 final worklists in memory
# -----------------------------
ROW_PRIMARY_DF = anom_top_rows_robust_sizeaware          # ANOM.2.a.1
ROW_ALT_DF     = anom_top_rows_mag_sizeaware             # ANOM.2.b.1
PROV_PRIMARY_DF= anom_top_providers_robust_sizeaware      # ANOM.3.a.1
PROV_ALT_DF    = anom_top_providers_mag_sizeaware         # ANOM.3.b.1

# -----------------------------
# 1) Timestamp + output directory
# -----------------------------
ts = pd.Timestamp.now(tz="America/New_York").strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("artifacts/anomaly_surfaces") / f"run_{ts}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 2) Parameter pack
# -----------------------------
# Row-level knobs
MIN_SLICE_N = 50
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2
TOP_N = 200

# ANOM.2.a.1 scoring weights
W_A1_LOG_OE_PCT = 0.6
W_A1_RESID_PCT  = 0.4
W_A1_HIGH_CONF  = 0.2

# ANOM.2.b.1 scoring weights + clip
LOG_CLIP_MAX = 2.0
W_B1_LOG_OE_PCT = 0.45
W_B1_RESID_PCT  = 0.35
W_B1_LOG_MAG    = 0.35
W_B1_HIGH_CONF  = 0.15

# Provider-level knobs
W_P_A1_CODES = 0.25
W_P_A1_YEARS = 0.25
W_P_B1_SEV   = 0.10

LOG_OE_Q = 0.995

# --- NEW: guard against missing/stale severity cut ---
if "log_oe_severity_cut" not in globals():
    raise NameError("Missing log_oe_severity_cut. Run ANOM.3.b.1 first (the cell that computes it).")

LOG_OE_SEVERITY_CUT = float(log_oe_severity_cut)

PARAMS = {
    "timestamp_et": ts,
    "outputs_dir": str(OUT_DIR),
    "row_level": {
        "slice_cols": ["HCPCS_Cd", "Year"],
        "MIN_SLICE_N": MIN_SLICE_N,
        "USE_MIN_EXPECTED_FILTER": USE_MIN_EXPECTED_FILTER,
        "MIN_EXPECTED": MIN_EXPECTED,
        "TOP_N": TOP_N,
        "ANOM_2_a_1": {
            "name": "robust + size-aware primary",
            "directionality": {"log_oe": "> 0", "residual": "> 0"},
            "score": f"{W_A1_LOG_OE_PCT}*log_oe_pct_in_slice + {W_A1_RESID_PCT}*resid_pct_in_slice + {W_A1_HIGH_CONF}*is_high_conf",
            "weights": {
                "log_oe_pct_in_slice": W_A1_LOG_OE_PCT,
                "resid_pct_in_slice": W_A1_RESID_PCT,
                "is_high_conf": W_A1_HIGH_CONF,
            },
            "filters": {
                "slice_n": f">= {MIN_SLICE_N}",
                "expected_cost": f">= {MIN_EXPECTED}" if USE_MIN_EXPECTED_FILTER else "none",
            },
            "output_df_name": "anom_top_rows_robust_sizeaware",
        },
        "ANOM_2_b_1": {
            "name": "magnitude-aware + size-aware alternative",
            "directionality": {"log_oe": "> 0", "residual": "> 0"},
            "log_mag_def": f"log_mag = clip(log_oe, 0, {LOG_CLIP_MAX}) / {LOG_CLIP_MAX}",
            "score": (
                f"{W_B1_LOG_OE_PCT}*log_oe_pct_in_slice + {W_B1_RESID_PCT}*resid_pct_in_slice + "
                f"{W_B1_LOG_MAG}*log_mag + {W_B1_HIGH_CONF}*is_high_conf"
            ),
            "weights": {
                "log_oe_pct_in_slice": W_B1_LOG_OE_PCT,
                "resid_pct_in_slice": W_B1_RESID_PCT,
                "log_mag": W_B1_LOG_MAG,
                "is_high_conf": W_B1_HIGH_CONF,
            },
            "filters": {
                "slice_n": f">= {MIN_SLICE_N}",
                "expected_cost": f">= {MIN_EXPECTED}" if USE_MIN_EXPECTED_FILTER else "none",
            },
            "output_df_name": "anom_top_rows_mag_sizeaware",
        },
    },
    "provider_level": {
        "group_key": ["Rndrng_NPI", "provider_type", "state"],
        "TOP_N": TOP_N,
        "ANOM_3_a_1": {
            "name": "repeat offenders (robust + size-aware)",
            "event_flag": (
                "is_row_anomalous_robust_sizeaware = "
                "(log_oe>0) & (residual>0) & (is_high_conf | high_confidence_anomaly_candidate) "
                "& (log_oe_pct_in_slice>=0.99) & (slice_n>=MIN_SLICE_N)"
            ),
            "provider_filter": "n_anom_rows_robust > 0",
            "provider_score": f"n_anom_rows_robust + {W_P_A1_CODES}*n_unique_codes + {W_P_A1_YEARS}*n_unique_years",
            "weights": {"n_unique_codes": W_P_A1_CODES, "n_unique_years": W_P_A1_YEARS},
            "output_df_name": "anom_top_providers_robust_sizeaware",
        },
        "ANOM_3_b_1": {
            "name": "shock providers (magnitude-aware + size-aware)",
            "severity_cut": {
                "q": LOG_OE_Q,
                "value": LOG_OE_SEVERITY_CUT,
                "definition": "quantile(log_oe, q) computed on BASE used in ANOM.3.b.1",
            },
            # --- FIXED: proper string concatenation + embeds literal cutoff value ---
            "event_flag": (
                "is_row_anomalous_mag_sizeaware = "
                "(log_oe>0) & (residual>0) & (is_high_conf | high_confidence_anomaly_candidate) "
                f"& (log_oe_pct_in_slice>=0.99) & (log_oe>={LOG_OE_SEVERITY_CUT}) & (slice_n>=MIN_SLICE_N)"
            ),
            "provider_score": (
                f"n_anom_rows_mag + 0.25*n_unique_codes + 0.25*n_unique_years + {W_P_B1_SEV}*p95_log_oe_mag"
            ),
            "weights": {"severity_bump_p95_log_oe_mag": W_P_B1_SEV},
            "output_df_name": "anom_top_providers_mag_sizeaware",
        },
    },
}

# -----------------------------
# 3) Helper: write df as parquet+csv with stable column order
# -----------------------------
def _write_df(df: pd.DataFrame, stem: str) -> dict:
    df = df.copy()

    cols = list(df.columns)
    if "row_id" in cols:
        cols = ["row_id"] + [c for c in cols if c != "row_id"]
        df = df[cols]

    parquet_path = OUT_DIR / f"{stem}__{ts}.parquet"
    csv_path     = OUT_DIR / f"{stem}__{ts}.csv"

    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)

    return {
        "stem": stem,
        "n_rows": int(len(df)),
        "n_cols": int(df.shape[1]),
        "parquet": str(parquet_path),
        "csv": str(csv_path),
    }

# -----------------------------
# 4) Validate presence + write all four worklists
# -----------------------------
required = {
    "rows_primary": ROW_PRIMARY_DF,
    "rows_alt": ROW_ALT_DF,
    "providers_primary": PROV_PRIMARY_DF,
    "providers_alt": PROV_ALT_DF,
}
for name, d in required.items():
    if d is None or not isinstance(d, pd.DataFrame):
        raise TypeError(f"{name} is missing or not a DataFrame. Check the variable names at the top of this cell.")

manifest = []
manifest.append(_write_df(ROW_PRIMARY_DF, "rows_primary__ANOM_2_a_1"))
manifest.append(_write_df(ROW_ALT_DF,     "rows_alt__ANOM_2_b_1"))
manifest.append(_write_df(PROV_PRIMARY_DF,"providers_primary__ANOM_3_a_1"))
manifest.append(_write_df(PROV_ALT_DF,    "providers_alt__ANOM_3_b_1"))

# -----------------------------
# 5) Write params + manifest
# -----------------------------
params_json_path = OUT_DIR / f"run_params__{ts}.json"
manifest_json_path = OUT_DIR / f"manifest__{ts}.json"
params_md_path = OUT_DIR / f"run_params__{ts}.md"

with open(params_json_path, "w") as f:
    json.dump(PARAMS, f, indent=2)

with open(manifest_json_path, "w") as f:
    json.dump({"timestamp_et": ts, "outputs_dir": str(OUT_DIR), "files": manifest}, f, indent=2)

md = []
md.append(f"# Anomaly surfacing worklists (frozen) — {ts} ET\n")
md.append(f"Output directory: `{OUT_DIR}`\n")
md.append("## Files written\n")
for item in manifest:
    md.append(f"- **{item['stem']}**: {item['n_rows']} × {item['n_cols']}\n"
              f"  - Parquet: `{item['parquet']}`\n"
              f"  - CSV: `{item['csv']}`\n")
md.append("\n## Parameters\n")
md.append("```json\n" + json.dumps(PARAMS, indent=2) + "\n```\n")

params_md_path.write_text("".join(md))

print("✅ Frozen 4 worklists to disk.")
print("Output dir:", OUT_DIR)
print("Wrote params:", params_json_path)
print("Wrote params md:", params_md_path)
print("Wrote manifest:", manifest_json_path)

display(pd.DataFrame(manifest))

# Post C4a patch sanity checks 

In [ ]:
import pandas as pd
import numpy as np

EVAL_PATH = "artifacts/eval_universe/eval_scored_DG_V3.parquet"
df = pd.read_parquet(
    EVAL_PATH,
    columns=["Year","HCPCS_Cd","oe_ratio","has_lag","expected_cost_support_tier","route"]
)

m = (
    (df["Year"] == 2020)
    & (df["HCPCS_Cd"].astype(str) == "J2469")
    & (df["expected_cost_support_tier"].astype(str) == "medium_high")
    & (df["route"].astype(str) == "cold_start")
    & (~df["has_lag"].astype(bool))
)

print("J2469/2020 medium_high cold_start cold rows:", int(m.sum()))
print("Median OE (should be ~1.03):", float(pd.to_numeric(df.loc[m, "oe_ratio"], errors="coerce").median()))